In [ ]:
import pandas as pd, numpy as np

# Phase 1 산출물 업로드 후 로딩
from google.colab import files

strat = pd.read_parquet("strat_final_inst.parquet")
print("로딩:", strat.shape)
print("컬럼:", list(strat.columns))
print("\nK 분포:", strat["K"].value_counts().to_dict())
print("K_state3 분포:", strat["K_state3"].value_counts().to_dict())
print("instability 결측:", strat["instability"].isna().sum())

로딩: (2000, 32)
컬럼: ['sample_id', 'dataset_id', 'fact_nature', 'construction_operator', 'reasoning_structure', 'provenance', 'desired_action', 'world_truth', 'context_internal_validity', 'benchmark_target', 'question', 'context', 'parametric_answer', 'contextual_answer', 'relation_type', 'answer_type', 'hop_count', 'domain', 'mod_degree_bin', 'num_edits', 's_pop', 'mulan_date', 'answer_candidates', 'cb_generation', 'K_state3', 'K', 'n_known', 'instability', 'K_final', 'prompt_tok_len', 'context_trunc', 'prompt_tok_len_trunc']

K 분포: {'K+': 1031, 'K-': 969}
K_state3 분포: {'K+': 1000, 'K-_other': 854, 'K-_refusal': 146}
instability 결측: 0


In [ ]:
# K-_refusal vs K-_other로 K- 하위군 분해
sub = strat[strat["K"]=="K-"].copy()

print("=== K- 하위군별 instability ===")
print(sub.groupby("K_state3")["instability"].agg(["mean","std","count"]))

print("\n=== 전체 K별 (기존 주장) ===")
print(strat.groupby("K")["instability"].agg(["mean","std","count"]))

# 핵심: refusal 빼면 K- instability가 얼마나 떨어지나
k_minus_no_refusal = sub[sub["K_state3"]=="K-_other"]["instability"].mean()
k_minus_all = sub["instability"].mean()
k_plus = strat[strat["K"]=="K+"]["instability"].mean()

print(f"\n=== 재현 주장 재검증 ===")
print(f"K+ instability:                 {k_plus:.3f}")
print(f"K- (전체, refusal 포함):        {k_minus_all:.3f}")
print(f"K- (refusal 제외, K-_other만):  {k_minus_no_refusal:.3f}")
print(f"→ refusal 제외 후에도 K+ < K- 유지되나? {'YES' if k_minus_no_refusal > k_plus else 'NO — 아티팩트!'}")

# refusal이 실제로 얼마나 부풀렸나
print(f"\nK-_refusal 평균 instability: {sub[sub['K_state3']=='K-_refusal']['instability'].mean():.3f}")
print(f"K-_refusal 개수: {(sub['K_state3']=='K-_refusal').sum()} / K- 전체 {len(sub)}")

=== K- 하위군별 instability ===
                mean       std  count
K_state3                             
K-_other    0.484569  0.281882    823
K-_refusal  0.724658  0.128954    146

=== 전체 K별 (기존 주장) ===
        mean       std  count
K                            
K+  0.274879  0.295302   1031
K-  0.520743  0.278115    969

=== 재현 주장 재검증 ===
K+ instability:                 0.275
K- (전체, refusal 포함):        0.521
K- (refusal 제외, K-_other만):  0.485
→ refusal 제외 후에도 K+ < K- 유지되나? YES

K-_refusal 평균 instability: 0.725
K-_refusal 개수: 146 / K- 전체 969


K− 0.52 중 일부는 refusal 아티팩트가 맞음 (146개가 0.725로 평균을 끌어올림)
근데 refusal 빼도 K-_other 0.485 vs K+ 0.275로 격차 0.21 유지 → 재현 유효
다만 앞으로 "K− 0.52 재현" 대신 "K-_other 0.485 재현"으로 보고하는 게 정확. refusal은 별도 취급.

In [ ]:
# 정확 중복
exact = strat["question"].value_counts()
dup_q = exact[exact > 1]
print("=== 정확히 중복된 질문 ===")
print(f"중복 질문 종류: {len(dup_q)}개")
print(f"중복에 연루된 샘플 수: {dup_q.sum()}개 / 2000")
print(f"고유 질문 수: {strat['question'].nunique()} / 2000")

print("\n=== dataset_id별 고유 질문 비율 ===")
for did, grp in strat.groupby("dataset_id"):
    print(f"  {did}: 고유 {grp['question'].nunique()} / {len(grp)} "
          f"({grp['question'].nunique()/len(grp)*100:.0f}%)")

# 중복 예시
if len(dup_q) > 0:
    print("\n=== 중복 질문 예시 (상위 3개) ===")
    for q in dup_q.head(3).index:
        rows = strat[strat["question"]==q]
        print(f"\n  Q: {q[:60]} ({len(rows)}회)")
        print(f"     dataset: {rows['dataset_id'].tolist()}")
        print(f"     contextual_answer: {rows['contextual_answer'].tolist()[:4]}")

=== 정확히 중복된 질문 ===
중복 질문 종류: 158개
중복에 연루된 샘플 수: 347개 / 2000
고유 질문 수: 1811 / 2000

=== dataset_id별 고유 질문 비율 ===
  clasheval: 고유 378 / 500 (76%)
  confiqa_qa: 고유 500 / 500 (100%)
  dynamicqa: 고유 497 / 500 (99%)
  nqswap: 고유 436 / 500 (87%)

=== 중복 질문 예시 (상위 3개) ===

  Q: What is the initial daily dosage of repotrectinib in mg for  (5회)
     dataset: ['clasheval', 'clasheval', 'clasheval', 'clasheval', 'clasheval']
     contextual_answer: ['240', '64', '192', '1600']

  Q: In which year did the Japanese girl vocal group Little Glee  (5회)
     dataset: ['clasheval', 'clasheval', 'clasheval', 'clasheval', 'clasheval']
     contextual_answer: ['1937', '2114', '1981', '1914']

  Q: Which year was the USS Missouri (BB-63), an Iowa-class battl (4회)
     dataset: ['clasheval', 'clasheval', 'clasheval', 'clasheval']
     contextual_answer: ['1933', '1999', '2021', '1955']


In [ ]:
clash = strat[strat["dataset_id"]=="clasheval"].copy()

# truncation된 context_trunc에 contextual_answer가 남아있는지
def ans_in_ctx(row):
    return str(row["contextual_answer"]) in str(row["context_trunc"])

clash["ans_preserved"] = clash.apply(ans_in_ctx, axis=1)

print("=== ClashEval 정답 보존 (truncation 후) ===")
print(clash["ans_preserved"].value_counts())
lost = clash[~clash["ans_preserved"]]
print(f"\n유실: {len(lost)}개")

# 유실된 게 원래 conflict였는지 — parametric도 없는지 확인
# (둘 다 없으면 conflict 조건 자체가 깨진 것)
def param_in_ctx(row):
    return str(row["parametric_answer"]) in str(row["context_trunc"])
lost = lost.copy()
lost["param_in_ctx"] = lost.apply(param_in_ctx, axis=1)

print("\n=== 유실 샘플 분석 ===")
print("contextual도 parametric도 없음 (conflict 완전 깨짐):",
      (~lost["param_in_ctx"]).sum())
print("parametric은 있음 (부분 유효):", lost["param_in_ctx"].sum())

# 원본 context엔 있었는데 truncation으로 잘린 건지 vs 원래부터 없던 건지
def ans_in_orig(row):
    return str(row["contextual_answer"]) in str(row["context"])
lost["in_original"] = lost.apply(ans_in_orig, axis=1)
print("\n원본엔 있었는데 truncation이 자름:", lost["in_original"].sum())
print("원본에도 없었음 (alias 표기 차이 등):", (~lost["in_original"]).sum())

=== ClashEval 정답 보존 (truncation 후) ===
ans_preserved
True     442
False     58
Name: count, dtype: int64

유실: 58개

=== 유실 샘플 분석 ===
contextual도 parametric도 없음 (conflict 완전 깨짐): 51
parametric은 있음 (부분 유효): 7

원본엔 있었는데 truncation이 자름: 18
원본에도 없었음 (alias 표기 차이 등): 40


In [ ]:
import re

def normalize_for_match(s):
    """숫자 표기·단위·대소문자 차이 흡수한 매칭용 정규화"""
    s = str(s).lower()
    s = s.replace(",", "")           # 1,600 → 1600
    s = re.sub(r"\s+", "", s)        # 240 mg → 240mg, 공백 제거
    s = re.sub(r"[^\w.]", "", s)     # 구두점 제거(소수점 . 는 유지)
    return s

def ans_in_ctx_norm(answer, context):
    return normalize_for_match(answer) in normalize_for_match(context)

# 아까 "원본에도 없음" 40개 재검사
clash = strat[strat["dataset_id"]=="clasheval"].copy()
clash["ans_preserved_orig"] = clash.apply(
    lambda r: str(r["contextual_answer"]) in str(r["context"]), axis=1)
orig_absent = clash[~clash["ans_preserved_orig"]]   # 원본 부재 40개

# 정규화 매칭으로 재검사
orig_absent = orig_absent.copy()
orig_absent["recovered_orig"] = orig_absent.apply(
    lambda r: ans_in_ctx_norm(r["contextual_answer"], r["context"]), axis=1)

print("=== 원본 부재 판정 40개 재검사 (정규화 매칭) ===")
print(f"정규화로 회복됨 (실은 표기차이): {orig_absent['recovered_orig'].sum()}")
print(f"여전히 부재 (진짜 없음): {(~orig_absent['recovered_orig']).sum()}")

# 회복된 것 예시
rec = orig_absent[orig_absent["recovered_orig"]]
if len(rec) > 0:
    print("\n=== 회복 예시 (표기 차이였던 것) ===")
    for _, r in rec.head(5).iterrows():
        print(f"  ans: {r['contextual_answer']!r}")
        # context에서 정답 주변 찾기
        cn = normalize_for_match(r["contextual_answer"])
        ctxn = normalize_for_match(r["context"])
        pos = ctxn.find(cn)
        print(f"     정규화 후 위치: {pos} (context 길이 {len(ctxn)})")

# 전체 truncation 보존율도 정규화로 재계산 (얼마나 개선되나)
clash["preserved_trunc_norm"] = clash.apply(
    lambda r: ans_in_ctx_norm(r["contextual_answer"], r["context_trunc"]), axis=1)
print(f"\n=== truncation 후 보존율 비교 ===")
print(f"기존 exact 매칭:   442/500")
print(f"정규화 매칭:       {clash['preserved_trunc_norm'].sum()}/500")

=== 원본 부재 판정 40개 재검사 (정규화 매칭) ===
정규화로 회복됨 (실은 표기차이): 4
여전히 부재 (진짜 없음): 36

=== 회복 예시 (표기 차이였던 것) ===
  ans: '10000'
     정규화 후 위치: 2208 (context 길이 6147)
  ans: '981600'
     정규화 후 위치: 1113 (context 길이 1179)
  ans: '82000'
     정규화 후 위치: 4264 (context 길이 5131)
  ans: '3000'
     정규화 후 위치: 4784 (context 길이 6688)

=== truncation 후 보존율 비교 ===
기존 exact 매칭:   442/500
정규화 매칭:       443/500


In [ ]:
import re

# 전체 풀 로딩 (재샘플링 소스)
pool = pd.read_parquet("conflict_pool_kstate.parquet")
pool["answer_candidates"] = pool["answer_candidates"].apply(
    lambda x: list(x) if isinstance(x, np.ndarray) else x)
print("풀 로딩:", pool.shape)

# --- 정규화 매칭 함수 (재샘플링 필터 + truncation 공용) ---
def normalize_for_match(s):
    s = str(s).lower().replace(",", "")
    s = re.sub(r"\s+", "", s)
    s = re.sub(r"[^\w.]", "", s)
    return s

def ans_in_context(answer, context):
    return normalize_for_match(answer) in normalize_for_match(context)

# --- 탐색 풀만 (mulan 제외, 기존과 동일) ---
pool_explore = pool[pool["dataset_id"] != "mulan"].copy()
pool_explore["K"] = pool_explore["K_state3"].apply(lambda x: "K+" if x=="K+" else "K-")

print("\n=== 탐색 풀 dataset별 크기 + 고유 질문 ===")
for did, grp in pool_explore.groupby("dataset_id"):
    uniq = grp["question"].nunique()
    print(f"  {did}: {len(grp)}행, 고유질문 {uniq} ({uniq/len(grp)*100:.0f}%)")

풀 로딩: (32614, 25)

=== 탐색 풀 dataset별 크기 + 고유 질문 ===
  clasheval: 10179행, 고유질문 1299 (13%)
  confiqa_qa: 6000행, 고유질문 5998 (100%)
  dynamicqa: 5689행, 고유질문 5277 (93%)
  nqswap: 4746행, 고유질문 1667 (35%)


In [ ]:
# question 단위 dedup: 각 질문당 1행만 (ClashEval은 mod_degree_bin 분포 보존하며)
def dedup_by_question(df, preserve_col=None, seed=42):
    """질문당 1행. preserve_col 있으면 그 분포 유지하며 랜덤 선택."""
    if preserve_col and preserve_col in df.columns:
        # 질문별로 preserve_col 값 다양성 유지 위해 랜덤 1개
        return df.sample(frac=1, random_state=seed).drop_duplicates("question")
    return df.sample(frac=1, random_state=seed).drop_duplicates("question")

# 정답 유실 필터 (정규화 매칭) — context에 정답 있는 것만
pool_explore["ans_ok"] = pool_explore.apply(
    lambda r: ans_in_context(r["contextual_answer"], r["context"])
    if pd.notna(r["contextual_answer"]) else True, axis=1)

print("=== 정답 유실 필터 후 dataset별 ===")
print(pool_explore.groupby("dataset_id")["ans_ok"].agg(["sum","count"]))

# 필터 통과 + dedup 후 셀(dataset×K)별 고유 질문 수
valid = pool_explore[pool_explore["ans_ok"]].copy()
deduped = dedup_by_question(valid)

print("\n=== dedup+필터 후 셀별 고유 질문 (재샘플링 가용량) ===")
cell = deduped.groupby(["dataset_id","K"]).size().reset_index(name="n_unique")
print(cell.to_string(index=False))
print(f"\n최소 셀: {cell['n_unique'].min()}")

# 셀당 250 가능한지
for target in [150, 200, 250]:
    ok = (cell["n_unique"] >= target).all()
    print(f"  셀당 {target}: {'가능' if ok else '불가'} (최소셀 {cell['n_unique'].min()})")

=== 정답 유실 필터 후 dataset별 ===
             sum  count
dataset_id             
clasheval   9522  10179
confiqa_qa  5967   6000
dynamicqa   5689   5689
nqswap      4746   4746

=== dedup+필터 후 셀별 고유 질문 (재샘플링 가용량) ===
dataset_id  K  n_unique
 clasheval K+       249
 clasheval K-      1027
confiqa_qa K+      2846
confiqa_qa K-      3110
 dynamicqa K+      1209
 dynamicqa K-      4060
    nqswap K+       741
    nqswap K-       926

최소 셀: 249
  셀당 150: 가능 (최소셀 249)
  셀당 200: 가능 (최소셀 249)
  셀당 250: 불가 (최소셀 249)


In [ ]:
np.random.seed(42)
TARGET = 240

# ClashEval은 mod_degree_bin 분포 보존하며 dedup 필요
# → question별로 뽑되, 전체 mod_degree_bin 분포를 최대한 유지
def stratified_dedup_sample(df, target, mod_col="mod_degree_bin", seed=42):
    """question 단위 dedup 후 target개 샘플. mod_col 있으면 분포 보존."""
    d = df.sample(frac=1, random_state=seed).drop_duplicates("question")
    if mod_col in d.columns and d[mod_col].notna().any():
        # mod_degree_bin 비율대로 층화 샘플
        frac = target / len(d)
        sampled = d.groupby(mod_col, group_keys=False).apply(
            lambda g: g.sample(max(1, int(round(len(g)*frac))), random_state=seed))
        # 목표 개수 맞춤 (반올림 오차 보정)
        if len(sampled) > target:
            sampled = sampled.sample(target, random_state=seed)
        elif len(sampled) < target:
            extra = d[~d.index.isin(sampled.index)].sample(
                target-len(sampled), random_state=seed)
            sampled = pd.concat([sampled, extra])
        return sampled
    return d.sample(min(target, len(d)), random_state=seed)

# 셀별 재샘플링
final_rows = []
for (did, k), grp in valid.groupby(["dataset_id","K"]):
    s = stratified_dedup_sample(grp, TARGET)
    final_rows.append(s)
    print(f"  {did} {k}: {len(s)}개 (고유질문 확인: {s['question'].nunique()})")

strat_new = pd.concat(final_rows).reset_index(drop=True)
print(f"\n총 {len(strat_new)}개")

# 검증: 전체 question 중복 없는지 (dataset 간 교차 중복도)
print("전체 고유 질문:", strat_new["question"].nunique(), "/", len(strat_new))
print("\n=== 셀 균형 ===")
print(strat_new.groupby(["dataset_id","K"]).size())

# ClashEval mod_degree_bin 분포 보존 확인
print("\n=== ClashEval mod_degree_bin 분포 (원본 vs 재샘플) ===")
orig_dist = valid[valid["dataset_id"]=="clasheval"]["mod_degree_bin"].value_counts(normalize=True)
new_dist = strat_new[strat_new["dataset_id"]=="clasheval"]["mod_degree_bin"].value_counts(normalize=True)
print(pd.DataFrame({"원본": orig_dist, "재샘플": new_dist}).round(3))

  clasheval K+: 240개 (고유질문 확인: 240)
  clasheval K-: 240개 (고유질문 확인: 240)
  confiqa_qa K+: 240개 (고유질문 확인: 240)
  confiqa_qa K-: 240개 (고유질문 확인: 240)
  dynamicqa K+: 240개 (고유질문 확인: 240)
  dynamicqa K-: 240개 (고유질문 확인: 240)
  nqswap K+: 240개 (고유질문 확인: 240)
  nqswap K-: 240개 (고유질문 확인: 240)

총 1920개
전체 고유 질문: 1916 / 1920

=== 셀 균형 ===
dataset_id  K 
clasheval   K+    240
            K-    240
confiqa_qa  K+    240
            K-    240
dynamicqa   K+    240
            K-    240
nqswap      K+    240
            K-    240
dtype: int64

=== ClashEval mod_degree_bin 분포 (원본 vs 재샘플) ===
                   원본    재샘플
mod_degree_bin              
large           0.170  0.150
medium          0.388  0.273
small           0.307  0.415
tiny            0.134  0.162


/tmp/ipykernel_4199/3000688147.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled = d.groupby(mod_col, group_keys=False).apply(
/tmp/ipykernel_4199/3000688147.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled = d.groupby(mod_col, group_keys=False).apply(


In [ ]:
# 전체에서 question 완전 dedup (dataset 교차 중복까지)
before = len(strat_new)
strat_new = strat_new.drop_duplicates("question").reset_index(drop=True)
print(f"교차 중복 제거: {before} → {len(strat_new)}")

# 제거로 셀이 240 밑으로 빠진 곳 있는지
print("\n=== 셀 균형 (교차 제거 후) ===")
print(strat_new.groupby(["dataset_id","K"]).size())

# 신규분 K-state 상태 확인 — 재검증(N=5) 안 된 K-_other가 몇 개인지
print("\n=== K_state3 분포 (신규 세트) ===")
print(strat_new["K_state3"].value_counts())
print("\n→ K-_other 중 N=5 재검증 필요:", (strat_new["K_state3"]=="K-_other").sum())
print("   (이건 Phase 2에서 모델 로딩 시 처리)")

# 최종 저장
strat_new.to_parquet("strat_v2_deduped.parquet", index=False)
print("\n✓ 저장: strat_v2_deduped.parquet")

교차 중복 제거: 1920 → 1916

=== 셀 균형 (교차 제거 후) ===
dataset_id  K 
clasheval   K+    240
            K-    240
confiqa_qa  K+    240
            K-    240
dynamicqa   K+    240
            K-    239
nqswap      K+    240
            K-    237
dtype: int64

=== K_state3 분포 (신규 세트) ===
K_state3
K+            960
K-_other      828
K-_refusal    128
Name: count, dtype: int64

→ K-_other 중 N=5 재검증 필요: 828
   (이건 Phase 2에서 모델 로딩 시 처리)

✓ 저장: strat_v2_deduped.parquet


In [ ]:
# 기존 strat_final_inst에 N=5 생성 원문이 있는지 확인
print("=== 컬럼에 샘플 생성 원문 있나 ===")
inst_cols = [c for c in strat.columns if any(k in c.lower()
             for k in ["sample","gen","instab"])]
print("관련 컬럼:", inst_cols)

print("\n=== cb_generation 내용 확인 (greedy 1회? N=5?) ===")
for i in range(3):
    print(f"  [{i}] {strat['cb_generation'].iloc[i]!r}")

# cb_generation이 문자열 1개면 greedy, 리스트면 N=5
sample_val = strat["cb_generation"].iloc[0]
print(f"\ncb_generation 타입: {type(sample_val)}")
print("→ N=5 원문 저장 여부:", "리스트(N=5 저장됨)" if isinstance(sample_val, (list, np.ndarray)) else "문자열(greedy 1회만, N=5 원문 없음)")

=== 컬럼에 샘플 생성 원문 있나 ===
관련 컬럼: ['sample_id', 'cb_generation', 'instability']

=== cb_generation 내용 확인 (greedy 1회? N=5?) ===
  [0] '10-20 units.'
  [1] '3:40.14 (set by Ian Thorpe in 2001)'
  [2] '335'

cb_generation 타입: <class 'str'>
→ N=5 원문 저장 여부: 문자열(greedy 1회만, N=5 원문 없음)


In [ ]:
import re

REFUSAL_PATTERNS = [
    "couldn't find", "could not find", "don't have", "do not have",
    "no information", "not sure", "i'm not aware", "i am not aware",
    "unable to", "cannot find", "can't find", "i don't know", "i do not know",
    "there is no", "not enough information", "insufficient", "no mention",
    "does not provide", "doesn't provide", "not specified", "not available",
    "cannot determine", "can't determine", "unclear", "no answer",
]

def is_refusal(text):
    """생성물이 거부/모름 응답인지. instability·behavior 공유."""
    t = str(text).lower().strip()
    if len(t) == 0:
        return True
    return any(p in t for p in REFUSAL_PATTERNS)

# 기존 cb_generation으로 테스트 (greedy지만 refusal 패턴 검증엔 충분)
strat_new = pd.read_parquet("strat_v2_deduped.parquet")
strat_new["is_refusal_greedy"] = strat_new["cb_generation"].apply(is_refusal)

print("=== refusal 판정 분포 (greedy 기준) ===")
print(strat_new["is_refusal_greedy"].value_counts())

print("\n=== K_state3별 refusal 비율 (K-_refusal과 일치해야 정상) ===")
print(strat_new.groupby("K_state3")["is_refusal_greedy"].mean())

print("\n=== refusal로 잡힌 예시 5개 ===")
for g in strat_new[strat_new["is_refusal_greedy"]]["cb_generation"].head(5):
    print(f"  {g[:70]!r}")
print("\n=== refusal 아닌데 K-_refusal인 것 (놓친 패턴) ===")
missed = strat_new[(~strat_new["is_refusal_greedy"]) & (strat_new["K_state3"]=="K-_refusal")]
for g in missed["cb_generation"].head(5):
    print(f"  {g[:70]!r}")

=== refusal 판정 분포 (greedy 기준) ===
is_refusal_greedy
False    1773
True      143
Name: count, dtype: int64

=== K_state3별 refusal 비율 (K-_refusal과 일치해야 정상) ===
K_state3
K+            0.015625
K-_other      0.000000
K-_refusal    1.000000
Name: is_refusal_greedy, dtype: float64

=== refusal로 잡힌 예시 5개 ===
  "I don't have information on a specific volcanic eruption in Iceland on"
  "I do not have information on Armando Bacot's points scored in the NCAA"
  'I do not have information on how many points Kyle Filipowski scored in'
  'I do not have information on the number of goals Jeff Skinner scored i'
  'I do not have information on how many points Chase Hunter scored in Cl'

=== refusal 아닌데 K-_refusal인 것 (놓친 패턴) ===


In [ ]:
print("=== ClashEval mod_degree_bin 분포 (버그 영향 확인) ===")
comp = pd.DataFrame({
    "dedup전(valid)": valid[valid.dataset_id=="clasheval"]["mod_degree_bin"].value_counts(normalize=True),
    "dedup후(현재)": strat_new[strat_new.dataset_id=="clasheval"]["mod_degree_bin"].value_counts(normalize=True),
}).round(3)
print(comp)
print("\n최대 편차:", (comp["dedup전(valid)"] - comp["dedup후(현재)"]).abs().max())

# 2번: 셀 구성 명시 기록
print("\n=== 최종 셀 구성 (기록용) ===")
print(strat_new.groupby(["dataset_id","K"]).size())
print("총:", len(strat_new))

=== ClashEval mod_degree_bin 분포 (버그 영향 확인) ===
                dedup전(valid)  dedup후(현재)
mod_degree_bin                           
large                   0.170       0.150
medium                  0.388       0.273
small                   0.307       0.415
tiny                    0.134       0.162

최대 편차: 0.11499999999999999

=== 최종 셀 구성 (기록용) ===
dataset_id  K 
clasheval   K+    240
            K-    240
confiqa_qa  K+    240
            K-    240
dynamicqa   K+    240
            K-    239
nqswap      K+    240
            K-    237
dtype: int64
총: 1916


In [ ]:
def stratified_dedup_v2(df, target, mod_col="mod_degree_bin", seed=42):
    """question 단위 dedup + mod_degree_bin 분포 보존 층화."""
    # 1) question당 1행 (그 질문이 가진 mod_bin 중 랜덤)
    d = df.sample(frac=1, random_state=seed).drop_duplicates("question")
    # 2) 원본(중복포함) 분포를 타겟으로 mod_bin별 할당량 계산
    if mod_col not in d.columns or d[mod_col].isna().all():
        return d.sample(min(target, len(d)), random_state=seed)
    target_frac = df[mod_col].value_counts(normalize=True)  # 원본 분포
    picks = []
    for b, frac in target_frac.items():
        pool_b = d[d[mod_col] == b]
        n_b = min(len(pool_b), int(round(target * frac)))
        if n_b > 0:
            picks.append(pool_b.sample(n_b, random_state=seed))
    out = pd.concat(picks) if picks else d.head(0)
    # 목표 수 보정
    if len(out) < target:
        extra = d[~d.index.isin(out.index)].sample(
            min(target-len(out), len(d)-len(out)), random_state=seed)
        out = pd.concat([out, extra])
    return out.head(target)

In [ ]:
np.random.seed(42)
TARGET = 240

# ClashEval만 v2로 재샘플링, 나머지는 mod_degree 없으니 기존 방식 유지
final_rows = []
for (did, k), grp in valid.groupby(["dataset_id","K"]):
    if did == "clasheval":
        s = stratified_dedup_v2(grp, TARGET)
    else:
        s = grp.sample(frac=1, random_state=42).drop_duplicates("question")
        s = s.sample(min(TARGET, len(s)), random_state=42)
    final_rows.append(s)

strat_new = pd.concat(final_rows).reset_index(drop=True)
# 교차 중복 제거
strat_new = strat_new.drop_duplicates("question").reset_index(drop=True)

# 검증
print("=== ClashEval mod_degree_bin 분포 재확인 ===")
comp = pd.DataFrame({
    "원본": valid[valid.dataset_id=="clasheval"]["mod_degree_bin"].value_counts(normalize=True),
    "재샘플v2": strat_new[strat_new.dataset_id=="clasheval"]["mod_degree_bin"].value_counts(normalize=True),
}).round(3)
print(comp)
print("최대 편차:", (comp["원본"]-comp["재샘플v2"]).abs().max())

print("\n=== 최종 셀 구성 ===")
print(strat_new.groupby(["dataset_id","K"]).size())
print("총:", len(strat_new), "| 고유질문:", strat_new["question"].nunique())

print("\n=== K_state3 분포 (신규 재검증 대상) ===")
print(strat_new["K_state3"].value_counts())

strat_new.to_parquet("strat_v2_deduped.parquet", index=False)
print("✓ 저장: strat_v2_deduped.parquet (덮어씀)")

=== ClashEval mod_degree_bin 분포 재확인 ===
                   원본  재샘플v2
mod_degree_bin              
large           0.170  0.162
medium          0.388  0.321
small           0.307  0.369
tiny            0.134  0.148
최대 편차: 0.067

=== 최종 셀 구성 ===
dataset_id  K 
clasheval   K+    240
            K-    240
confiqa_qa  K+    240
            K-    240
dynamicqa   K+    240
            K-    239
nqswap      K+    240
            K-    237
dtype: int64
총: 1916 | 고유질문: 1916

=== K_state3 분포 (신규 재검증 대상) ===
K_state3
K+            960
K-_other      820
K-_refusal    136
Name: count, dtype: int64
✓ 저장: strat_v2_deduped.parquet (덮어씀)


[신규 1916개 — context_trunc 없음, instability 없음, 신호 없음]

        ↓

① truncation (정규화 매칭 기반, 전체 재적용)     ← 피드백 3

        ↓

② closed-book N=5 (temp 0.7) 생성 원문 저장

        ↓

③ 같은 생성물로 통합 처리:                        ← 피드백 4
   - K-_other → K+ 승격 재판정 (≥3 known)
   - instability = non-refusal 샘플만 (refusal 제외)  ← 피드백 5 함수
   - refusal_rate 별도 컬럼

        ↓

④ K 최종 확정 → 셀 재확인 (240 유지되나)

        ↓

⑤ 신호 추출 (open-book, context 넣음):
   - generation-time attention (per-head, ctx/q/prev-gen, 생성토큰 평균)
   - pre-answer attention (비교용, 128d)
   - hidden state (답변 위치)
   - behavior 라벨 (open-book greedy → contextual/parametric/other/abstain)

In [ ]:
def is_refusal(text, head_chars=100):
    """첫 head_chars만 검사 — '...unclear, but it was 1994' 같은 거짓양성 방지"""
    t = str(text).lower().strip()
    if len(t) == 0:
        return True
    head = t[:head_chars]
    return any(p in head for p in REFUSAL_PATTERNS)



In [ ]:
!pip install -q scikit-learn

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm.auto import tqdm
from huggingface_hub import login
login()

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.float16, device_map="cuda",
    attn_implementation="eager",
)
model.eval()
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "left"
print("model loaded | layers:", model.config.num_hidden_layers,
      "| heads:", model.config.num_attention_heads)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

model loaded | layers: 32 | heads: 32


In [ ]:
def truncate_context(context, answer, tok, max_ctx_tokens=400):
    """정답 중심 윈도우. 정규화 매칭으로 정답 위치 탐색 (거짓 부재 방지)."""
    ctx_ids = tok(context, add_special_tokens=False)["input_ids"]
    if len(ctx_ids) <= max_ctx_tokens:
        return context

    # 정규화 매칭으로 정답 위치 찾기
    ans_norm = normalize_for_match(answer)
    # 원문에서 정답 char 위치 추정: 정규화 공간에서 찾고 원문 비율로 역산
    ctx_norm = normalize_for_match(context)
    pos_norm = ctx_norm.find(ans_norm)

    if pos_norm < 0:
        # 정규화로도 못 찾음 → 앞에서 자름
        return tok.decode(ctx_ids[:max_ctx_tokens], skip_special_tokens=True)

    # 정규화 공간 위치 → 원문 비율 → 토큰 위치
    ans_tok_pos = int(len(ctx_ids) * pos_norm / max(len(ctx_norm), 1))
    half = max_ctx_tokens // 2
    start = max(0, ans_tok_pos - half)
    end = min(len(ctx_ids), start + max_ctx_tokens)
    start = max(0, end - max_ctx_tokens)
    return tok.decode(ctx_ids[start:end], skip_special_tokens=True)

# 전체 적용
strat_new["context_trunc"] = [
    truncate_context(ex["context"], ex["contextual_answer"], tok)
    for _, ex in tqdm(strat_new.iterrows(), total=len(strat_new))
]

# 검증: 정답 보존율 (정규화 매칭 기준)
strat_new["ans_preserved"] = strat_new.apply(
    lambda r: ans_in_context(r["contextual_answer"], r["context_trunc"])
    if pd.notna(r["contextual_answer"]) else True, axis=1)

# 프롬프트 길이 재확인
def build_conflict_prompt_parts(question, context):
    msgs = [
        {"role": "system", "content": "Answer the question based on the given context."},
        {"role": "user", "content": f"Context: {context}\n\nQuestion: {question}\nAnswer:"},
    ]
    full = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return full, None

strat_new["prompt_len"] = [
    len(tok(build_conflict_prompt_parts(ex["question"], ex["context_trunc"])[0],
            add_special_tokens=False)["input_ids"])
    for _, ex in strat_new.iterrows()
]

print("=== truncation 후 ===")
print("정답 보존:", strat_new["ans_preserved"].sum(), "/", len(strat_new))
print("\n프롬프트 길이:")
print(strat_new["prompt_len"].describe())
print("\ndataset별 길이 중앙값:")
print(strat_new.groupby("dataset_id")["prompt_len"].median())

  0%|          | 0/1916 [00:00<?, ?it/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


=== truncation 후 ===
정답 보존: 1906 / 1916

프롬프트 길이:
count    1916.000000
mean      250.910752
std       143.603366
min        62.000000
25%       146.750000
50%       179.000000
75%       459.250000
max       519.000000
Name: prompt_len, dtype: float64

dataset별 길이 중앙값:
dataset_id
clasheval     474.5
confiqa_qa    163.0
dynamicqa     143.0
nqswap        194.0
Name: prompt_len, dtype: float64


In [ ]:
import re, string
import time

def normalize_answer(s):
    s = str(s).lower()
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = "".join(ch for ch in s if ch not in string.punctuation)
    return re.sub(r"\s+", " ", s).strip()

def is_known(gen, candidates):
    gn = normalize_answer(gen)
    if not gn: return False
    return any(normalize_answer(c) and normalize_answer(c) in gn for c in candidates)

REFUSAL_PATTERNS = [
    "couldn't find","could not find","don't have","do not have","no information",
    "not sure","i'm not aware","i am not aware","unable to","cannot find","can't find",
    "i don't know","i do not know","there is no","not enough information","insufficient",
    "no mention","does not provide","doesn't provide","not specified","not available",
    "cannot determine","can't determine","no answer",
]
def is_refusal(text, head_chars=100):
    t = str(text).lower().strip()
    if len(t)==0: return True
    return any(p in t[:head_chars] for p in REFUSAL_PATTERNS)

def build_closedbook_prompt(q):
    msgs = [{"role":"system","content":"Answer the question concisely with only the answer, no explanation."},
            {"role":"user","content":f"Question: {q}\nAnswer:"}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def sample_batch(questions, n_samples=5, batch_size=48, max_new=32, temp=0.7):
    expanded, owner = [], []
    for qi, q in enumerate(questions):
        for _ in range(n_samples):
            expanded.append(q); owner.append(qi)
    results = [[] for _ in questions]
    for i in tqdm(range(0, len(expanded), batch_size)):
        batch = [build_closedbook_prompt(q) for q in expanded[i:i+batch_size]]
        owners = owner[i:i+batch_size]
        enc = tok(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to("cuda")
        gen = model.generate(**enc, max_new_tokens=max_new, do_sample=True,
                             temperature=temp, top_p=0.9, pad_token_id=tok.eos_token_id)
        for j in range(len(batch)):
            new = gen[j][enc["input_ids"].shape[1]:]
            results[owners[j]].append(tok.decode(new, skip_special_tokens=True).strip())
    return results

def compute_instability_norefusal(samples):
    """non-refusal 샘플만으로 instability. refusal_rate 별도 반환."""
    from collections import Counter
    n = len(samples)
    non_ref = [s for s in samples if not is_refusal(s)]
    refusal_rate = 1.0 - len(non_ref)/n if n else 0.0
    if len(non_ref) <= 1:
        return np.nan, refusal_rate  # 계산 불가
    norm = [normalize_answer(s) for s in non_ref]
    modal_count = Counter(norm).most_common(1)[0][1]
    return 1.0 - modal_count/len(non_ref), refusal_rate

# === 전체 1916개 N=5 ===
t0 = time.time()
all_samples = sample_batch(strat_new["question"].tolist(), n_samples=5, batch_size=48)
print(f"\nN=5 완료: {(time.time()-t0)/60:.1f}분")

# 통합 처리
strat_new["cb_samples"] = all_samples  # 원문 저장 (검수용)
strat_new["n_known5"] = [sum(is_known(s,c) for s in samp)
                         for samp,c in zip(all_samples, strat_new["answer_candidates"])]
inst_ref = [compute_instability_norefusal(s) for s in all_samples]
strat_new["instability"] = [x[0] for x in inst_ref]
strat_new["refusal_rate"] = [x[1] for x in inst_ref]

# K 재확정: 원 K+ 유지 + K-_other 중 ≥3 known 승격
def final_k(row):
    if row["K_state3"]=="K+": return "K+"
    if row["n_known5"]>=3: return "K+"   # 승격
    return "K-"
strat_new["K"] = strat_new.apply(final_k, axis=1)

print("\n=== K 재확정 ===")
print(strat_new["K"].value_counts())
print("승격(K-_other→K+):", ((strat_new["K_state3"]=="K-_other")&(strat_new["K"]=="K+")).sum())
print("\n=== instability (refusal 제외) K별 ===")
print(strat_new.groupby("K")["instability"].agg(["mean","std","count"]))
print("\ninstability 결측(non-refusal≤1):", strat_new["instability"].isna().sum())
print("refusal_rate 평균:", strat_new["refusal_rate"].mean().round(3))

  0%|          | 0/200 [00:00<?, ?it/s]


N=5 완료: 14.0분

=== K 재확정 ===
K
K+    982
K-    934
Name: count, dtype: int64
승격(K-_other→K+): 20

=== instability (refusal 제외) K별 ===
        mean       std  count
K                            
K+  0.274286  0.291510    969
K-  0.480655  0.276474    840

instability 결측(non-refusal≤1): 107
refusal_rate 평균: 0.068


In [ ]:
# 결측 107개가 어디 쏠렸나
miss = strat_new[strat_new["instability"].isna()]
print("=== instability 결측 107개 분포 ===")
print("K별:", miss["K"].value_counts().to_dict())
print("K_state3별:", miss["K_state3"].value_counts().to_dict())
print("dataset별:", miss["dataset_id"].value_counts().to_dict())

# 눈 검수: refusal 판정이 temp 0.7에서 제대로 되나 (거짓 refusal 확인)
print("\n=== refusal_rate 높은 샘플의 생성물 (판정 검수) ===")
high_ref = strat_new[strat_new["refusal_rate"]>=0.6].head(3)
for _, r in high_ref.iterrows():
    print(f"\n[{r['dataset_id']}] Q: {r['question'][:50]}")
    for s in r["cb_samples"]:
        print(f"   ref={is_refusal(s)} | {s[:65]!r}")

# 반대: refusal_rate 0인데 instability 높은 것 (정상 답변 다양성인지)
print("\n=== instability 높은데 refusal 0인 샘플 (진짜 불안정) ===")
high_inst = strat_new[(strat_new["refusal_rate"]==0)&(strat_new["instability"]>=0.6)].head(2)
for _, r in high_inst.iterrows():
    print(f"\n[{r['dataset_id']}] Q: {r['question'][:50]} | inst={r['instability']:.2f}")
    for s in r["cb_samples"]:
        print(f"   {s[:60]!r}")

=== instability 결측 107개 분포 ===
K별: {'K-': 94, 'K+': 13}
K_state3별: {'K-_refusal': 91, 'K+': 12, 'K-_other': 4}
dataset별: {'clasheval': 71, 'dynamicqa': 27, 'nqswap': 7, 'confiqa_qa': 2}

=== refusal_rate 높은 샘플의 생성물 (판정 검수) ===

[clasheval] Q: How many points did KJ Simpson score in the NCAA T
   ref=True | "I don't have information on a player named KJ Simpson scoring in "
   ref=True | "I don't have access to real-time information or specific details "
   ref=True | 'I do not have information on the score of the specific NCAA Tourn'
   ref=True | "I am unable to verify KJ Simpson's score in the NCAA Tournament g"
   ref=True | "I don't have information about KJ Simpson's score in the NCAA Tou"

[clasheval] Q: How many points did Tristen Newton score in the NC
   ref=True | 'I do not have information on the points scored by Tristen Newton '
   ref=True | 'I do not have information on the points scored by Tristen Newton '
   ref=True | "I do not have information on Tristen Newton's poin

In [ ]:
# 패턴 추가 (검수에서 발견된 누락)
REFUSAL_PATTERNS += ["cannot verify", "can't verify", "cannot access",
                     "no access", "unable to access", "cannot provide",
                     "can't provide", "cannot confirm", "can't confirm"]

# cb_samples로 재계산 (GPU 불필요)
inst_ref2 = [compute_instability_norefusal(s) for s in strat_new["cb_samples"]]
strat_new["instability"] = [x[0] for x in inst_ref2]
strat_new["refusal_rate"] = [x[1] for x in inst_ref2]

print("=== 패턴 보강 후 재계산 ===")
print("instability 결측:", strat_new["instability"].isna().sum(), "(이전 107)")
print("refusal_rate 평균:", strat_new["refusal_rate"].mean().round(3), "(이전 0.068)")

# 핵심: K+ vs K-_other 격차 유지 확인 (refusal 순수 제거 후)
print("\n=== instability K별 (refusal 제외, 패턴 보강) ===")
print(strat_new.groupby("K")["instability"].agg(["mean","std","count"]))

# K-_other만 따로 (헤드라인 수치)
ko = strat_new[strat_new["K_state3"]=="K-_other"]["instability"].mean()
kp = strat_new[strat_new["K"]=="K+"]["instability"].mean()
print(f"\n헤드라인: K+ {kp:.3f} vs K-_other {ko:.3f} | 격차 {ko-kp:.3f}")
print("→ 격차 유지?", "YES" if ko-kp > 0.15 else "약화됨 — 재검토")

=== 패턴 보강 후 재계산 ===
instability 결측: 115 (이전 107)
refusal_rate 평균: 0.072 (이전 0.068)

=== instability K별 (refusal 제외, 패턴 보강) ===
        mean       std  count
K                            
K+  0.274053  0.291571    968
K-  0.479212  0.277583    833

헤드라인: K+ 0.274 vs K-_other 0.480 | 격차 0.206
→ 격차 유지? YES


In [ ]:
# K+인데 instability 결측 (non-refusal ≤1)
kp_miss = strat_new[(strat_new["K"]=="K+") & (strat_new["instability"].isna())]
print(f"=== K+ 결측 {len(kp_miss)}개 검수 ===")
for _, r in kp_miss.head(12).iterrows():
    print(f"\n[{r['dataset_id']}] Q: {r['question'][:55]}")
    print(f"   greedy: {r['cb_generation'][:55]!r} | gold: {r['parametric_answer']}")
    print(f"   refusal_rate: {r['refusal_rate']:.1f}")
    for s in r["cb_samples"][:5]:
        print(f"      ref={is_refusal(s)} | {s[:50]!r}")

=== K+ 결측 14개 검수 ===

[clasheval] Q: How many points did KJ Simpson score in the NCAA Tourna
   greedy: "I don't have information on KJ Simpson's NCAA Tournamen" | gold: 23
   refusal_rate: 1.0
      ref=True | "I don't have information on a player named KJ Simp"
      ref=True | "I don't have access to real-time information or sp"
      ref=True | 'I do not have information on the score of the spec'
      ref=True | "I am unable to verify KJ Simpson's score in the NC"
      ref=True | "I don't have information about KJ Simpson's score "

[clasheval] Q: How many points did Tristen Newton score in the NCAA To
   greedy: "I do not have information on Tristen Newton's points in" | gold: 20
   refusal_rate: 1.0
      ref=True | 'I do not have information on the points scored by '
      ref=True | 'I do not have information on the points scored by '
      ref=True | "I do not have information on Tristen Newton's poin"
      ref=True | "I don't have information on that specific game."
      

In [ ]:
# is_known에 refusal 가드 추가: refusal이면 무조건 unknown
def is_known_guarded(gen, candidates):
    if is_refusal(gen):        # refusal 답변은 안다고 볼 수 없음
        return False
    return is_known(gen, candidates)

# n_known5 재계산 (refusal 가드 적용)
strat_new["n_known5"] = [
    sum(is_known_guarded(s, c) for s in samp)
    for samp, c in zip(strat_new["cb_samples"], strat_new["answer_candidates"])
]

# K_state3의 K+도 재검증 필요 — greedy가 refusal인데 K+였던 것 확인
strat_new["greedy_refusal"] = strat_new["cb_generation"].apply(is_refusal)
bad_kplus = strat_new[(strat_new["K_state3"]=="K+") & (strat_new["greedy_refusal"])]
print("K_state3=K+인데 greedy가 refusal (원본 오분류):", len(bad_kplus))

# K 재확정: refusal 가드 반영
def final_k_v2(row):
    # 원 K+였어도 greedy가 refusal이면 재검증 대상으로
    if row["K_state3"]=="K+" and not row["greedy_refusal"]:
        return "K+"
    if row["n_known5"]>=3:   # refusal 가드된 known 카운트
        return "K+"
    return "K-"
strat_new["K"] = strat_new.apply(final_k_v2, axis=1)

print("\n=== K 재확정 (refusal 가드) ===")
print(strat_new["K"].value_counts())
print("이전 K+ 982 → 현재:", (strat_new["K"]=="K+").sum())

# 헤드라인 재확인 — K+ 오염 제거 후 격차
print("\n=== instability K별 (오염 제거 후) ===")
print(strat_new.groupby("K")["instability"].agg(["mean","std","count"]))
kp = strat_new[strat_new["K"]=="K+"]["instability"].mean()
km_other = strat_new[(strat_new["K"]=="K-")&(strat_new["K_state3"]=="K-_other")]["instability"].mean()
print(f"\nK+ {kp:.3f} vs K-_other {km_other:.3f} | 격차 {km_other-kp:.3f}")

K_state3=K+인데 greedy가 refusal (원본 오분류): 15

=== K 재확정 (refusal 가드) ===
K
K+    966
K-    950
Name: count, dtype: int64
이전 K+ 982 → 현재: 966

=== instability K별 (오염 제거 후) ===
        mean      std  count
K                           
K+  0.273223  0.29128    966
K-  0.479681  0.27744    835

K+ 0.273 vs K-_other 0.476 | 격차 0.203


In [ ]:
# 결측 처리 방침: abstain 그룹 분리, imputation 안 함 (피드백 3)
# instability NaN인 샘플 = abstain 지배 그룹 → 클러스터링 instability축에서 제외, refusal_rate가 신호
strat_new["is_abstain_group"] = strat_new["instability"].isna()

print("=== 최종 라벨 상태 ===")
print("K:", strat_new["K"].value_counts().to_dict())
print("abstain 그룹(instability 결측):", strat_new["is_abstain_group"].sum())
print("  └ K별:", strat_new[strat_new["is_abstain_group"]].groupby("K").size().to_dict())

print("\n=== 최종 셀 구성 ===")
print(strat_new.groupby(["dataset_id","K"]).size())

# 저장 (cb_samples 리스트는 str로)
strat_new["cb_samples_str"] = strat_new["cb_samples"].apply(lambda x: " ||| ".join(x))
save_cols = [c for c in strat_new.columns if c != "cb_samples"]
strat_new[save_cols].to_parquet("strat_v2_final.parquet", index=False)
print("\n✓ 저장: strat_v2_final.parquet")

=== 최종 라벨 상태 ===
K: {'K+': 966, 'K-': 950}
abstain 그룹(instability 결측): 115
  └ K별: {'K-': 115}

=== 최종 셀 구성 ===
dataset_id  K 
clasheval   K+    229
            K-    251
confiqa_qa  K+    246
            K-    234
dynamicqa   K+    241
            K-    238
nqswap      K+    250
            K-    227
dtype: int64

✓ 저장: strat_v2_final.parquet


피드백 반영 항목: 2(셀 구성 기록), 3(truncation 흐름 누락), 4(재검증 통합 순서), 5(temp 0.7 거짓 refusal). 모델 로딩 전 1·2 확인.

2·3·4 반영 내용: 2(v2 편차/셀 기록 문서화), 3(결측 abstain 그룹 분리 + K+ 결측 12개 검수), 4(숫자 instability 인플레 플래그).

In [ ]:
import re

def is_known_v2(gen, candidates):
    """refusal 가드 + word-boundary 매칭 (숫자 substring 오염 차단)"""
    if is_refusal(gen):
        return False
    g = str(gen)
    for c in candidates:
        cs = str(c).strip()
        if not cs:
            continue
        # word-boundary regex (숫자·엔티티 정확 매칭)
        if re.search(rf"(?<!\w){re.escape(cs)}(?!\w)", g, re.IGNORECASE):
            return True
    return False

# n_known5 재계산 (word-boundary)
strat_new["n_known5"] = [
    sum(is_known_v2(s, c) for s in samp)
    for samp, c in zip(strat_new["cb_samples"], strat_new["answer_candidates"])
]

# greedy도 word-boundary로 재판정 (K_state3 원본 오염 확인)
strat_new["greedy_known_v2"] = [
    is_known_v2(g, c) for g, c in zip(strat_new["cb_generation"], strat_new["answer_candidates"])
]

# K 재확정 (word-boundary + refusal 가드)
def final_k_v3(row):
    if row["greedy_known_v2"]:      # greedy가 (정확)정답이면 K+
        return "K+"
    if row["n_known5"] >= 3:        # N=5 다수 (정확)known 승격
        return "K+"
    return "K-"
strat_new["K"] = strat_new.apply(final_k_v3, axis=1)

# K_state3 재유도 (1번: 보강 refusal 기준으로 통일)
strat_new["greedy_refusal"] = strat_new["cb_generation"].apply(is_refusal)
def kstate3_v2(row):
    if row["K"] == "K+":
        return "K+"
    if row["greedy_refusal"]:
        return "K-_refusal"
    return "K-_other"
strat_new["K_state3"] = strat_new.apply(kstate3_v2, axis=1)

print("=== word-boundary 재확정 후 ===")
print("K:", strat_new["K"].value_counts().to_dict())
print("K_state3:", strat_new["K_state3"].value_counts().to_dict())

print("\n=== dataset별 K+ 변화 (ClashEval 관전) ===")
print(strat_new.groupby("dataset_id")["K"].apply(lambda x: (x=="K+").sum()))

print("\n=== 헤드라인 재확인 ===")
print(strat_new.groupby("K")["instability"].agg(["mean","std","count"]))
ko = strat_new[strat_new["K_state3"]=="K-_other"]["instability"]
kp = strat_new[strat_new["K"]=="K+"]["instability"]
print(f"\nK+ {kp.mean():.3f} (n={kp.count()}) vs K-_other {ko.mean():.3f} (n={ko.count()})")
print(f"격차 {ko.mean()-kp.mean():.3f}")

=== word-boundary 재확정 후 ===
K: {'K-': 968, 'K+': 948}
K_state3: {'K+': 948, 'K-_other': 810, 'K-_refusal': 158}

=== dataset별 K+ 변화 (ClashEval 관전) ===
dataset_id
clasheval     222
confiqa_qa    244
dynamicqa     235
nqswap        247
Name: K, dtype: int64

=== 헤드라인 재확인 ===
        mean       std  count
K                            
K+  0.269339  0.289733    948
K-  0.479640  0.277917    853

K+ 0.269 (n=948) vs K-_other 0.475 (n=809)
격차 0.206


라벨 확정. 이번 재작업에서 수정한 버그:

- mod_degree 층화 no-op
- refusal 패턴 누락 (cannot verify류)
- refusal 답변 substring K+ 오분류
- 숫자 gold plain-substring 오염 (word-boundary로 수정)

남은 작업: 3·4번(문서 기록) — 저장 시 함께 처리.

In [ ]:
# --- 4번: 장부 정합성 확인 ---
print("=== 장부 확인 ===")
print("최종 K+:", (strat_new["K"]=="K+").sum(), "/ K-:", (strat_new["K"]=="K-").sum())
print("셀 구성:")
print(strat_new.groupby(["dataset_id","K"]).size())

# --- abstain 그룹 (instability 결측) ---
strat_new["is_abstain_group"] = strat_new["instability"].isna()
print("\nabstain 그룹(instability 결측):", strat_new["is_abstain_group"].sum())
print("  └ K별:", strat_new[strat_new["is_abstain_group"]].groupby("K").size().to_dict())

# --- 저장 (라벨 확정본) ---
strat_new["cb_samples_str"] = strat_new["cb_samples"].apply(lambda x: " ||| ".join(x))
save_cols = [c for c in strat_new.columns if c != "cb_samples"]
strat_new[save_cols].to_parquet("strat_v2_final.parquet", index=False)
print("\n✓ 저장: strat_v2_final.parquet")

# --- 3번: 풀 라벨 오염 경고를 메타로 남김 ---
provenance_note = {
    "kstate_label_version": "v2",
    "kstate_logic": "refusal_guard + word_boundary_match",
    "pool_label_warning": "conflict_pool_kstate.parquet의 K_state3는 v1(가드없음). 재샘플 시 재판정 필수",
    "bugs_fixed": ["mod_degree_noop", "refusal_pattern_missing",
                   "refusal_substring_Kplus", "numeric_plain_substring"],
    "headline": "K+ 0.269 vs K-_other 0.475, gap 0.206",
}
import json
with open("label_provenance.json","w") as f:
    json.dump(provenance_note, f, ensure_ascii=False, indent=2)
print("✓ 저장: label_provenance.json")

=== 장부 확인 ===
최종 K+: 948 / K-: 968
셀 구성:
dataset_id  K 
clasheval   K+    222
            K-    258
confiqa_qa  K+    244
            K-    236
dynamicqa   K+    235
            K-    244
nqswap      K+    247
            K-    230
dtype: int64

abstain 그룹(instability 결측): 115
  └ K별: {'K-': 115}

✓ 저장: strat_v2_final.parquet
✓ 저장: label_provenance.json


## Phase 2 시작

In [ ]:
def build_neutral_prompt(question, context):
    """중립 프롬프트 — '기반 답변' 지시 없이 context+question만"""
    msgs = [{"role": "user",
             "content": f"Context: {context}\n\nQuestion: {question}\nAnswer:"}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def get_segment_bounds_neutral(question, context):
    full = build_neutral_prompt(question, context)
    enc = tok(full, add_special_tokens=False, return_offsets_mapping=True)
    offsets = enc["offset_mapping"]
    ctx_char = full.find("Context:")
    q_char = full.find("Question:")
    ans_char = full.rfind("Answer:")
    def c2t(cp):
        for i,(s,e) in enumerate(offsets):
            if s <= cp < e or s >= cp: return i
        return len(offsets)
    return {"n_tokens": len(offsets),
            "ctx_range": (c2t(ctx_char), c2t(q_char)),
            "q_range": (c2t(q_char), c2t(ans_char))}

@torch.no_grad()
def extract_gentime_attention(question, context, max_new=64):
    """답 토큰 생성하며 스텝별 attention을 3구간(ctx/q/prev-gen)으로 running mean."""
    b = get_segment_bounds_neutral(question, context)
    full = build_neutral_prompt(question, context)
    enc = tok(full, return_tensors="pt", add_special_tokens=False).to("cuda")
    prompt_len = enc["input_ids"].shape[1]
    ctx_s, ctx_e = b["ctx_range"]; q_s, q_e = b["q_range"]

    gen = model.generate(**enc, max_new_tokens=max_new, do_sample=False,
                         pad_token_id=tok.eos_token_id,
                         output_attentions=True, return_dict_in_generate=True)

    # gen.attentions: tuple(len=생성스텝수), 각 스텝은 tuple(len=32layer) [1,head,q,kv]
    n_steps = len(gen.attentions)
    n_layers = model.config.num_hidden_layers
    n_heads = model.config.num_attention_heads

    # running sum: [layer, head, 3구간(ctx/q/prevgen)]
    acc = np.zeros((n_layers, n_heads, 3), dtype=np.float64)
    for step in range(n_steps):
        cur_pos = prompt_len + step  # 현재 생성 토큰의 절대 위치
        for L in range(n_layers):
            # 이 스텝, 이 layer: [1, head, 1(현재토큰), kv_len]
            a = gen.attentions[step][L][0, :, -1, :]  # [head, kv_len]
            ctx_mass = a[:, ctx_s:ctx_e].sum(-1)      # [head]
            q_mass = a[:, q_s:q_e].sum(-1)
            prevgen_mass = a[:, prompt_len:cur_pos].sum(-1) if cur_pos > prompt_len else np.zeros(n_heads)
            acc[L, :, 0] += ctx_mass.float().cpu().numpy()
            acc[L, :, 1] += q_mass.float().cpu().numpy()
            acc[L, :, 2] += (prevgen_mass.float().cpu().numpy() if torch.is_tensor(prevgen_mass) else prevgen_mass)
    acc /= max(n_steps, 1)  # running mean
    gen_text = tok.decode(gen.sequences[0][prompt_len:], skip_special_tokens=True)
    return acc.astype(np.float32), n_steps, gen_text

# 파일럿 5개
for i in range(5):
    ex = strat_new.iloc[i]
    feat, nsteps, gtext = extract_gentime_attention(ex["question"], ex["context_trunc"])
    print(f"[{ex['dataset_id']}] steps={nsteps} | shape={feat.shape} | gen={gtext[:45]!r}")

print("\nflatten 차원:", feat.flatten().shape, "(= 32×32×3 = 3072 예상)")
print("샘플당 저장:", feat.nbytes/1024, "KB")

[clasheval] steps=37 | shape=(32, 32, 3) | gen="The year Phil Knight's memoir, 'Shoe Dog', wa"
[clasheval] steps=29 | shape=(32, 32, 3) | gen='The USS Missouri (BB-63), an Iowa-class battl'
[clasheval] steps=4 | shape=(32, 32, 3) | gen='1954.'
[clasheval] steps=21 | shape=(32, 32, 3) | gen='Ciara gave birth to her first child, Future Z'
[clasheval] steps=64 | shape=(32, 32, 3) | gen='The year Wayne Arthurs and Paul Hanley won th'

flatten 차원: (3072,) (= 32×32×3 = 3072 예상)
샘플당 저장: 12.0 KB


In [ ]:
import gc
# 재실행 전 초기화 (안전)
gentime_feats, gentime_meta, fail = [], [], []

In [ ]:
t0 = time.time()
gentime_feats = []
gentime_meta = []
fail = []

for i, (_, ex) in enumerate(tqdm(strat_new.iterrows(), total=len(strat_new))):
    try:
        feat, nsteps, gtext = extract_gentime_attention(ex["question"], ex["context_trunc"])
        gentime_feats.append(feat.flatten())   # [3072]
        gentime_meta.append({"sample_id": ex["sample_id"], "n_steps": nsteps,
                             "open_book_gen": gtext})
    except Exception as e:
        fail.append((ex["sample_id"], str(e)[:60]))
        gentime_feats.append(np.full(3072, np.nan, dtype=np.float32))
        gentime_meta.append({"sample_id": ex["sample_id"], "n_steps": 0, "open_book_gen": ""})
    if i % 100 == 0 and i > 0:
        gc.collect(); torch.cuda.empty_cache()

print(f"\n완료: {(time.time()-t0)/60:.1f}분, 실패 {len(fail)}")
if fail: print(fail[:5])

gentime_arr = np.stack(gentime_feats)  # [1916, 3072]
meta_df = pd.DataFrame(gentime_meta)
np.save("gentime_attention.npy", gentime_arr)
np.save("gentime_sample_ids.npy", meta_df["sample_id"].values)
meta_df.to_parquet("gentime_meta.parquet", index=False)

print("✓ gentime_attention.npy:", gentime_arr.shape)
print("NaN 샘플:", np.isnan(gentime_arr).any(axis=1).sum())
print("생성길이 분포:", meta_df["n_steps"].describe()[["mean","50%","max"]].to_dict())

  0%|          | 0/1916 [00:00<?, ?it/s]


완료: 77.5분, 실패 0
✓ gentime_attention.npy: (1916, 3072)
NaN 샘플: 0
생성길이 분포: {'mean': 32.35020876826722, '50%': 29.0, 'max': 64.0}


In [ ]:
# meta의 open_book_gen을 strat에 병합
meta_df = pd.read_parquet("gentime_meta.parquet")
strat_new = strat_new.merge(meta_df[["sample_id","open_book_gen","n_steps"]],
                            on="sample_id", how="left")

def classify_behavior(row):
    """context 넣고 생성한 답(open_book_gen)이 무엇을 따랐나.
    is_known_v2 계열로 매칭 (숫자 오염 방지)."""
    gen = row["open_book_gen"]
    if is_refusal(gen):
        return "abstain"
    ctx_ans = row["contextual_answer"]
    par_ans = row["parametric_answer"]
    # is_known_v2와 동일 매칭 (word-boundary)
    def match(ans):
        if pd.isna(ans): return False
        return bool(re.search(rf"(?<!\w){re.escape(str(ans))}(?!\w)", str(gen), re.IGNORECASE))
    hit_ctx = match(ctx_ans)
    hit_par = match(par_ans)
    if hit_ctx and not hit_par:
        return "accepted_context"      # 문맥 따름
    if hit_par and not hit_ctx:
        return "retained_parametric"   # 자기 지식 고수
    if hit_ctx and hit_par:
        return "both"                  # 둘 다 언급
    return "other"                     # 제3의 답

strat_new["behavior"] = strat_new.apply(classify_behavior, axis=1)

print("=== behavior 분포 ===")
print(strat_new["behavior"].value_counts())

print("\n=== desired_action × behavior (핵심 교차) ===")
print(pd.crosstab(strat_new["desired_action"], strat_new["behavior"]))

print("\n=== dataset_id × behavior ===")
print(pd.crosstab(strat_new["dataset_id"], strat_new["behavior"]))

=== behavior 분포 ===
behavior
accepted_context       847
other                  345
both                   318
retained_parametric    294
abstain                112
Name: count, dtype: int64

=== desired_action × behavior (핵심 교차) ===
behavior            abstain  accepted_context  both  other  \
desired_action                                               
accept                   66               364   184    188   
needs_verification       37               198   102     65   
reject                    9               285    32     92   

behavior            retained_parametric  
desired_action                           
accept                              155  
needs_verification                   77  
reject                               62  

=== dataset_id × behavior ===
behavior    abstain  accepted_context  both  other  retained_parametric
dataset_id                                                             
clasheval         9               285    32     92                   62

**behavior 라벨 결과**

핵심 관찰 — 모델은 desired_action을 잘 따르지 않음:

- reject 조건(clasheval)에서 accepted_context 285개 (retained_parametric 62개의 4.6배). 거짓 문맥을 거부해야 하는데 대부분 문맥을 따름. ClashEval의 원래 발견(모델이 거짓 context에 취약)과 일치.
- accept 조건에서도 accepted_context 364 vs retained_parametric 155 — 따르긴 하지만 고수도 상당수.

→ desired_action(벤치가 원하는 행동)과 model_action(실제 행동)이 크게 어긋남. 라벨을 두 축으로 분리한 근거가 됨. 클러스터가 어느 축과 정렬되는지가 핵심 정보.

주의 — both(318)와 other(345)가 많음. 합계 663개(35%). 클러스터링에서 애매한 그룹이 될 수 있음:

- both: contextual과 parametric이 답에 모두 등장 (예: "문맥은 X라지만 실제론 Y")
- other: 제3의 답 (둘 다 아님)

both가 많은 이유는 open_book_gen이 문장형이라 생성이 길어지면서 양쪽 답을 모두 언급했기 때문일 수 있음. 이후 진단에서 감안.

In [ ]:
# behavior 포함해서 중간 저장
strat_new.to_parquet("strat_v2_behavior.parquet", index=False)
print("✓ behavior 저장")

@torch.no_grad()
def extract_preanswer_hidden(question, context):
    """pre-answer(마지막 프롬프트 토큰) attention 128d + hidden 131072d. forward 1회."""
    b = get_segment_bounds_neutral(question, context)
    full = build_neutral_prompt(question, context)
    enc = tok(full, return_tensors="pt", add_special_tokens=False).to("cuda")
    out = model(**enc, output_attentions=True, output_hidden_states=True, use_cache=False)

    ctx_s, ctx_e = b["ctx_range"]; q_s, q_e = b["q_range"]
    last = enc["input_ids"].shape[1] - 1

    # pre-answer attention 128d (head 평균, 기존과 동일 — 비교용)
    attn_feats = []
    for L in range(len(out.attentions)):
        al = out.attentions[L][0][:, last, :].mean(0)
        ctx_mass = al[ctx_s:ctx_e].sum().item()
        q_mass = al[q_s:q_e].sum().item()
        ctx_len = max(ctx_e-ctx_s,1)
        ca = al[ctx_s:ctx_e]; ca = ca/(ca.sum()+1e-9)
        ce = -(ca*(ca+1e-9).log()).sum().item()
        attn_feats += [ctx_mass, q_mass, ctx_mass/ctx_len, ce]

    # hidden 131072d (답변 위치, layer 1~32)
    hid = np.concatenate([out.hidden_states[L][0, last, :].float().cpu().numpy()
                          for L in range(1, len(out.hidden_states))])
    return np.array(attn_feats, dtype=np.float32), hid.astype(np.float32)

t0 = time.time()
pre_attn, hidden_mat, sids = [], [], []
for i, (_, ex) in enumerate(tqdm(strat_new.iterrows(), total=len(strat_new))):
    a, h = extract_preanswer_hidden(ex["question"], ex["context_trunc"])
    pre_attn.append(a); hidden_mat.append(h); sids.append(ex["sample_id"])
    if i % 100 == 0 and i > 0:
        gc.collect(); torch.cuda.empty_cache()

print(f"\n완료: {(time.time()-t0)/60:.1f}분")

np.save("preanswer_attention.npy", np.stack(pre_attn))      # [1916, 128]
np.save("hidden_features.npy", np.stack(hidden_mat).astype(np.float32))  # [1916, 131072]
np.save("signal_sample_ids.npy", np.array(sids))
print("✓ pre-answer:", np.stack(pre_attn).shape)
print("✓ hidden:", np.stack(hidden_mat).shape)

✓ behavior 저장


  0%|          | 0/1916 [00:00<?, ?it/s]


완료: 4.6분
✓ pre-answer: (1916, 128)
✓ hidden: (1916, 131072)


In [ ]:
from google.colab import files
for f in ["gentime_attention.npy", "preanswer_attention.npy", "hidden_features.npy",
          "signal_sample_ids.npy", "gentime_sample_ids.npy", "strat_v2_behavior.parquet"]:
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
def match_any(gen, answers):
    """alias 리스트 전체를 word-boundary로 매칭 (K-state와 동일 기준)"""
    if not isinstance(answers, (list, np.ndarray)):
        answers = [answers]
    return any(pd.notna(a) and re.search(rf"(?<!\w){re.escape(str(a))}(?!\w)", str(gen), re.IGNORECASE)
               for a in answers if str(a).strip())

# 2번: ctx==par (충돌 없는 샘플) 먼저 식별
def is_no_conflict(row):
    ca, pa = row["contextual_answer"], row["parametric_answer"]
    if pd.isna(ca) or pd.isna(pa):
        return False
    return normalize_for_match(ca) == normalize_for_match(pa)

strat_new["no_conflict"] = strat_new.apply(is_no_conflict, axis=1)
print("=== ctx==par (충돌 없음) 샘플 ===")
print("개수:", strat_new["no_conflict"].sum())
print(pd.crosstab(strat_new["no_conflict"], strat_new["behavior"]))

# behavior 재계산 (alias + no_conflict 분리)
def classify_behavior_v2(row):
    gen = row["open_book_gen"]
    if is_refusal(gen):
        return "abstain"
    if row["no_conflict"]:
        return "no_conflict"   # 비교 무의미 — 별도 분리
    hit_ctx = match_any(gen, row["contextual_answer"])
    hit_par = match_any(gen, row["answer_candidates"])  # ← alias 리스트 사용
    if hit_ctx and not hit_par:
        return "accepted_context"
    if hit_par and not hit_ctx:
        return "retained_parametric"
    if hit_ctx and hit_par:
        return "both"
    return "other"

strat_new["behavior"] = strat_new.apply(classify_behavior_v2, axis=1)

print("\n=== behavior 재계산 (alias + no_conflict) ===")
print(strat_new["behavior"].value_counts())

# 3번: K × behavior (연구질문 핵심 축)
print("\n=== K × behavior (핵심) ===")
print(pd.crosstab(strat_new["K"], strat_new["behavior"]))

# 4번: K_state3 × behavior (abstain 코어 확인)
print("\n=== K_state3 × behavior (abstain 정합) ===")
print(pd.crosstab(strat_new["K_state3"], strat_new["behavior"]))

=== ctx==par (충돌 없음) 샘플 ===
개수: 0
behavior     abstain  accepted_context  both  other  retained_parametric
no_conflict                                                             
False            112               847   318    345                  294

=== behavior 재계산 (alias + no_conflict) ===
behavior
accepted_context       821
both                   344
other                  320
retained_parametric    319
abstain                112
Name: count, dtype: int64

=== K × behavior (핵심) ===
behavior  abstain  accepted_context  both  other  retained_parametric
K                                                                    
K+             37               313   261     99                  238
K-             75               508    83    221                   81

=== K_state3 × behavior (abstain 정합) ===
behavior    abstain  accepted_context  both  other  retained_parametric
K_state3                                                               
K+               37               313   

In [ ]:
other_rows = strat_new[strat_new["behavior"]=="other"]
print("=== other dataset별 분포 ===")
print(other_rows["dataset_id"].value_counts())

print("\n=== other 예시 10개 (진짜 제3의 답인지) ===")
for _, r in other_rows.head(10).iterrows():
    print(f"\n[{r['dataset_id']}] Q: {r['question'][:45]}")
    print(f"   gen: {r['open_book_gen'][:60]!r}")
    print(f"   ctx_ans: {r['contextual_answer']} | par: {r['parametric_answer']}")

=== other dataset별 분포 ===
dataset_id
nqswap        98
clasheval     92
confiqa_qa    65
dynamicqa     65
Name: count, dtype: int64

=== other 예시 10개 (진짜 제3의 답인지) ===

[clasheval] Q: What is the daily dosage of alemtuzumab for t
   gen: 'Unfortunately, the provided information does not specify the'
   ctx_ans: 90 | par: 30

[clasheval] Q: In which year did Sam Baldock join Derby Coun
   gen: '2022'
   ctx_ans: 1966 | par: 2021

[clasheval] Q: What is the initial daily dosage of allopurin
   gen: 'The initial daily dosage of allopurinol recommended for the '
   ctx_ans: 20 | par: 100

[clasheval] Q: What is the daily dosage of ombitasvir for pa
   gen: 'For patients with genotype 1b chronic hepatitis C, without c'
   ctx_ans: 75 | par: 25

[clasheval] Q: How many points did KJ Simpson score in the N
   gen: 'The article does not mention the date of the game, but it me'
   ctx_ans: 9 | par: 23

[clasheval] Q: What is the daily dosage in mg for the treatm
   gen: 'The text does not specifi

In [ ]:
# other 중 생성이 잘렸는지 (마지막 문자가 문장부호 아니면 잘림 의심)
other_rows = strat_new[strat_new["behavior"]=="other"].copy()
other_rows["gen_len"] = other_rows["open_book_gen"].str.len()
other_rows["truncated"] = ~other_rows["open_book_gen"].str.rstrip().str.endswith((".","!","?",'"'))

print("=== other 생성 길이/잘림 ===")
print("생성 길이 분포:", other_rows["gen_len"].describe()[["mean","50%","max"]].to_dict())
print("잘린 것으로 의심(문장 미완결):", other_rows["truncated"].sum(), "/", len(other_rows))

# does not mention/specify류가 refusal 놓친 건지
no_info = other_rows["open_book_gen"].str.lower().str.contains(
    "does not|doesn't|not mention|not specif|not provide|no information|cannot", regex=True)
print("'정보 없음'류 (abstain 놓침):", no_info.sum())

# n_steps=64(max 도달=잘림 확정)인 other 비율
merged = other_rows.merge(meta_df[["sample_id","n_steps"]], on="sample_id", how="left", suffixes=("","_m"))
print("n_steps=64 (생성 max 도달=잘림):", (merged["n_steps"]>=64).sum() if "n_steps" in merged else "n_steps 없음")

=== other 생성 길이/잘림 ===
생성 길이 분포: {'mean': 153.6875, '50%': 130.5, 'max': 350.0}
잘린 것으로 의심(문장 미완결): 81 / 320
'정보 없음'류 (abstain 놓침): 75
n_steps=64 (생성 max 도달=잘림): 72


잘림 81개 (문장 미완결) + n_steps=64 도달 72개 — 대체로 겹침, 생성이 max에서 잘려 정답 도달 전 끝난 것. 매칭 실패지 진짜 other 아님.
'정보 없음'류 75개 — open-book인데 "문서에 없다"고 함. 사실상 abstain인데 놓침 (첫 100자 밖이라 is_refusal 못 잡음).
나머지 ~160여개가 진짜 제3의 답.

In [ ]:
# is_refusal을 첫 100자 제한 없이 전체 검사하는 버전 (behavior용)
# — instability는 첫 100자 유지(생성 시작만 봄), behavior는 전체 문장에서 거부 표현 탐지
NO_INFO_PATTERNS = REFUSAL_PATTERNS + [
    "does not specify", "does not mention", "doesn't specify", "doesn't mention",
    "does not provide", "not provided in", "not mentioned in", "not specified in",
    "no information about", "does not contain", "doesn't contain",
]
def is_abstain_full(text):
    """behavior용: 전체 문장에서 거부/정보없음 탐지"""
    t = str(text).lower()
    return any(p in t for p in NO_INFO_PATTERNS)

# other 중 '정보 없음'류만 abstain으로 재분류
def classify_behavior_v3(row):
    gen = row["open_book_gen"]
    if is_abstain_full(gen):        # 전체 문장 기준 거부/정보없음
        return "abstain"
    if row["no_conflict"]:
        return "no_conflict"
    hit_ctx = match_any(gen, row["contextual_answer"])
    hit_par = match_any(gen, row["answer_candidates"])
    if hit_ctx and not hit_par:  return "accepted_context"
    if hit_par and not hit_ctx:  return "retained_parametric"
    if hit_ctx and hit_par:      return "both"
    return "other"

strat_new["behavior"] = strat_new.apply(classify_behavior_v3, axis=1)

print("=== behavior 최종 (정보없음→abstain) ===")
print(strat_new["behavior"].value_counts())

print("\n=== K × behavior 최종 ===")
print(pd.crosstab(strat_new["K"], strat_new["behavior"]))

# 남은 other 재확인 (진짜 제3의 답만 남았는지)
rem = strat_new[strat_new["behavior"]=="other"]
print(f"\n남은 other: {len(rem)}개")
print("잘림 비율:", (~rem["open_book_gen"].str.rstrip().str.endswith((".","!","?",'"'))).sum(), "/", len(rem))

# behavior 최종 저장
strat_new.to_parquet("strat_v2_behavior.parquet", index=False)
print("✓ 저장")

=== behavior 최종 (정보없음→abstain) ===
behavior
accepted_context       763
both                   331
retained_parametric    300
other                  264
abstain                258
Name: count, dtype: int64

=== K × behavior 최종 ===
behavior  abstain  accepted_context  both  other  retained_parametric
K                                                                    
K+             91               298   252     81                  226
K-            167               465    79    183                   74

남은 other: 264개
잘림 비율: 58 / 264
✓ 저장


In [ ]:
from google.colab import files
for f in ["gentime_attention.npy", "preanswer_attention.npy", "hidden_features.npy",
          "signal_sample_ids.npy", "strat_v2_behavior.parquet"]:
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
ab = strat_new[strat_new["behavior"]=="abstain"]
ab_hit = ab.apply(lambda r: match_any(r["open_book_gen"], r["contextual_answer"])
                  or match_any(r["open_book_gen"], r["answer_candidates"]), axis=1)
print("=== abstain 중 hedged (거부 표현 + 답도 있음) ===")
print(f"hedged: {ab_hit.sum()} / {len(ab)}")
print("\n=== hedged 예시 5개 ===")
for g in ab[ab_hit]["open_book_gen"].head(5):
    print(f"  {g[:120]!r}")

# K별로도 (K+ abstain이 hedged인지)
print("\n=== abstain의 K × hedged ===")
ab2 = ab.copy(); ab2["hedged"] = ab_hit.values
print(pd.crosstab(ab2["K"], ab2["hedged"]))

=== abstain 중 hedged (거부 표현 + 답도 있음) ===
hedged: 127 / 258

=== hedged 예시 5개 ===
  "Unfortunately, the provided information does not specify the exact year when 'Game Party: In Motion' debuted during the "
  'The information provided does not mention the year Yuri Danilochkin competed at the Winter Olympics in Sochi. However, b'
  'The year in which Godwin Obaseki switched from the APC to the PDP is not mentioned in the given text. However, it is men'
  'The information provided does not mention the Old Åsane Church being set on fire and sustaining heavy damage on Christma'
  'The text does not mention the date of the game, but it mentions the game was in the NCAA Tournament. However, it does me'

=== abstain의 K × hedged ===
hedged  False  True 
K                   
K+         40     51
K-         91     76


In [ ]:
def classify_behavior_v4(row):
    gen = row["open_book_gen"]
    hit_ctx = match_any(gen, row["contextual_answer"])
    hit_par = match_any(gen, row["answer_candidates"])
    hedged = is_abstain_full(gen)   # 거부/불확실 표현 존재 여부 (플래그용)

    # 답이 있으면 답 우선 (hedged여도)
    if hit_ctx and not hit_par:  beh = "accepted_context"
    elif hit_par and not hit_ctx: beh = "retained_parametric"
    elif hit_ctx and hit_par:     beh = "both"
    elif hedged:                  beh = "abstain"   # 답 없고 거부 표현만 → 진짜 abstain
    else:                         beh = "other"
    return pd.Series({"behavior": beh, "hedged": hedged and (hit_ctx or hit_par)})

res = strat_new.apply(classify_behavior_v4, axis=1)
strat_new["behavior"] = res["behavior"]
strat_new["hedged"] = res["hedged"]

print("=== behavior v4 (답 우선 + hedged 분리) ===")
print(strat_new["behavior"].value_counts())
print("\nhedged 플래그 (답 있으나 불확실 표현):", strat_new["hedged"].sum())

# 2번: no_conflict 확인
print("\n=== no_conflict ===")
print("ctx==par:", strat_new["no_conflict"].sum(), "(0이면 반사실 벤치라 정상)")

# 3번: K × behavior 최종
print("\n=== K × behavior 최종 ===")
print(pd.crosstab(strat_new["K"], strat_new["behavior"]))

# 3번: K_state3 × behavior (abstain 코어)
print("\n=== K_state3 × behavior ===")
print(pd.crosstab(strat_new["K_state3"], strat_new["behavior"]))

# 4번: dataset 내부에서도 K 패턴 유지되나 (조성 confound 체크)
print("\n=== dataset × K → behavior 비율 (조성 confound 체크) ===")
print(pd.crosstab([strat_new["dataset_id"], strat_new["K"]], strat_new["behavior"],
                  normalize="index").round(2))

=== behavior v4 (답 우선 + hedged 분리) ===
behavior
accepted_context       846
both                   352
retained_parametric    323
other                  264
abstain                131
Name: count, dtype: int64

hedged 플래그 (답 있으나 불확실 표현): 127

=== no_conflict ===
ctx==par: 0 (0이면 반사실 벤치라 정상)

=== K × behavior 최종 ===
behavior  abstain  accepted_context  both  other  retained_parametric
K                                                                    
K+             40               322   265     81                  240
K-             91               524    87    183                   83

=== K_state3 × behavior ===
behavior    abstain  accepted_context  both  other  retained_parametric
K_state3                                                               
K+               40               322   265     81                  240
K-_other         66               417    84    168                   75
K-_refusal       25               107     3     15                    8

=== dataset × 

In [ ]:
strat_new.to_parquet("strat_v2_behavior.parquet", index=False)
print("✓ 저장: strat_v2_behavior.parquet")
print("behavior:", strat_new["behavior"].value_counts().to_dict())
print("hedged:", strat_new["hedged"].sum())

✓ 저장: strat_v2_behavior.parquet
behavior: {'accepted_context': 846, 'both': 352, 'retained_parametric': 323, 'other': 264, 'abstain': 131}
hedged: 127


In [ ]:
from google.colab import files

for f in [
    "gentime_attention.npy",       # [1916, 3072] generation-time
    "preanswer_attention.npy",     # [1916, 128] pre-answer
    "hidden_features.npy",         # [1916, 131072] hidden
    "signal_sample_ids.npy",       # 신호 행 순서 (hidden/preanswer용)
    "gentime_sample_ids.npy",      # gentime 행 순서
    "gentime_meta.parquet",        # n_steps, open_book_gen
    "strat_v2_behavior.parquet",   # 라벨+instability+behavior+hedged (최종)
    "label_provenance.json",       # 버그/함수맵 기록
]:
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Phase 3

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, HDBSCAN
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# 현재 메모리 변수 확인
print("gentime_arr:", gentime_arr.shape)          # [1916, 3072]
# preanswer/hidden은 np.stack 안 하고 리스트로 있을 수 있음 — 확인
print("pre_attn 타입:", type(pre_attn), "len:", len(pre_attn) if isinstance(pre_attn, list) else pre_attn.shape)
print("hidden_mat 타입:", type(hidden_mat), "len:", len(hidden_mat) if isinstance(hidden_mat, list) else hidden_mat.shape)

# stack (리스트면)
preanswer = np.stack(pre_attn) if isinstance(pre_attn, list) else pre_attn
hidden = np.stack(hidden_mat).astype(np.float32) if isinstance(hidden_mat, list) else hidden_mat

# 순서 확인: gentime은 meta_df["sample_id"] 순서, preanswer/hidden은 sids 순서
print("\ngentime 순서(meta_df) == preanswer 순서(sids)?:",
      list(meta_df["sample_id"]) == list(sids))

# strat_new를 신호 순서(sids)에 맞춰 정렬
labels = strat_new.set_index("sample_id").loc[sids].reset_index()
print("라벨 정렬:", labels.shape, "| K:", labels["K"].value_counts().to_dict())

# gentime도 같은 순서인지 재확인 후, 다르면 재정렬
if list(meta_df["sample_id"]) != list(sids):
    order = [list(meta_df["sample_id"]).index(s) for s in sids]
    gentime_arr = gentime_arr[order]
    print("⚠️ gentime 재정렬함")
else:
    print("✓ 모든 신호 동일 순서")

gentime_arr: (1916, 3072)
pre_attn 타입: <class 'list'> len: 1916
hidden_mat 타입: <class 'list'> len: 1916

gentime 순서(meta_df) == preanswer 순서(sids)?: True
라벨 정렬: (1916, 43) | K: {'K-': 968, 'K+': 948}
✓ 모든 신호 동일 순서


In [ ]:
# K-/K+ 마스크
mask_kn = (labels["K"]=="K-").values
mask_kp = (labels["K"]=="K+").values

def cluster_signal(X, name, n_clusters_range=[2,3,4,5], pca_dim=30):
    """표준화 → PCA → KMeans. 라벨 없이 순수 신호만."""
    Xs = StandardScaler().fit_transform(X)
    n_pca = min(pca_dim, Xs.shape[0]-1, Xs.shape[1])
    Xp = PCA(n_components=n_pca, random_state=42).fit_transform(Xs)

    from sklearn.metrics import silhouette_score
    print(f"=== {name} (n={len(X)}, PCA {n_pca}d) ===")
    results = {}
    for k in n_clusters_range:
        km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(Xp)
        sil = silhouette_score(Xp, km.labels_)
        results[k] = km.labels_
        print(f"  k={k}: silhouette={sil:.3f}")
    return Xp, results

# K- 먼저
gentime_kn = gentime_arr[mask_kn]
Xp_kn, clust_kn = cluster_signal(gentime_kn, "gentime K-")

# PCA 설명력도 확인
Xs = StandardScaler().fit_transform(gentime_kn)
pca = PCA(n_components=30, random_state=42).fit(Xs)
print(f"\nPCA 누적 설명분산 (상위10): {pca.explained_variance_ratio_[:10].cumsum().round(3)}")

=== gentime K- (n=968, PCA 30d) ===
  k=2: silhouette=0.229
  k=3: silhouette=0.215
  k=4: silhouette=0.216
  k=5: silhouette=0.212

PCA 누적 설명분산 (상위10): [0.309 0.423 0.528 0.576 0.602 0.627 0.646 0.662 0.675 0.686]


In [ ]:
kn_labels = labels[mask_kn].reset_index(drop=True)
c2 = clust_kn[2]  # k=2 클러스터 라벨

def diagnose(cluster_labels, meta_df, axes):
    print(f"클러스터 크기: {np.bincount(cluster_labels)}")
    for ax in axes:
        vals = meta_df[ax].fillna("NaN").astype(str).values
        ari = adjusted_rand_score(vals, cluster_labels)
        nmi = normalized_mutual_info_score(vals, cluster_labels)
        print(f"  {ax:20s} ARI={ari:+.3f}  NMI={nmi:.3f}")

print("=== gentime K- (k=2) 정렬 진단 ===")
diagnose(c2, kn_labels,
         ["dataset_id", "desired_action", "behavior", "construction_operator",
          "fact_nature", "world_truth"])

# 연속 confound는 상관으로
print("\n=== 연속 confound (클러스터별 평균) ===")
for col in ["prompt_len", "instability", "n_steps"]:
    if col in kn_labels.columns:
        m0 = kn_labels[c2==0][col].mean()
        m1 = kn_labels[c2==1][col].mean()
        print(f"  {col:15s} c0={m0:.3f}  c1={m1:.3f}  diff={abs(m0-m1):.3f}")

# dataset 분포를 클러스터별로 (아티팩트 확인)
print("\n=== 클러스터 × dataset (아티팩트 경보) ===")
print(pd.crosstab(c2, kn_labels["dataset_id"]))

=== gentime K- (k=2) 정렬 진단 ===
클러스터 크기: [460 508]
  dataset_id           ARI=+0.041  NMI=0.041
  desired_action       ARI=+0.048  NMI=0.044
  behavior             ARI=+0.027  NMI=0.035
  construction_operator ARI=+0.015  NMI=0.018
  fact_nature          ARI=-0.002  NMI=0.007
  world_truth          ARI=-0.002  NMI=0.007

=== 연속 confound (클러스터별 평균) ===
  prompt_len      c0=237.643  c1=263.187  diff=25.544
  instability     c0=0.490  c1=0.470  diff=0.020
  n_steps         c0=46.043  c1=18.817  diff=27.227

=== 클러스터 × dataset (아티팩트 경보) ===
dataset_id  clasheval  confiqa_qa  dynamicqa  nqswap
row_0                                               
0                  77         129         98     156
1                 181         107        146      74


**gentime k=2 클러스터 해석**

1. 라벨 축과 거의 정렬되지 않음. dataset_id ARI 0.041, behavior 0.027, desired_action 0.048 — 전부 0.05 미만. gentime k=2 클러스터는 conflict 메커니즘·행동·출처 어느 것도 아님. (dataset ARI가 낮은 것은 출처 아티팩트가 아니라는 점에서는 긍정적이나, 의미 있는 축도 아님.)

2. 클러스터를 만든 실제 축은 n_steps. c0=46.0 vs c1=18.8, diff 27.2. generation-time attention이 생성 길이로 갈림 — 긴 답(c0) vs 짧은 답(c1).

running mean 방식의 구조적 아티팩트일 가능성이 큼. 생성 스텝이 많으면(긴 답) prev-gen 구간 attention이 누적되어 feature 성격이 달라지고, 짧은 답은 prev-gen이 거의 없음. 즉 클러스터는 conflict가 아니라 답 길이(n_steps)를 반영.

prompt_len diff(25.5)도 있으나 n_steps(27.2)가 더 크며, 둘 다 내용이 아닌 형식 특징.

**판정**: gentime k=2는 n_steps 아티팩트 = 표면 특징으로 갈린 전형적 사례. behavior/dataset/mechanism 어느 것도 잡히지 않았으므로, generation-time attention은 (적어도 running-mean 형태로는) conflict 신호를 저차원 클러스터로 드러내지 않음.

대응 방안:

- (A) n_steps 회귀 제거 후 재클러스터 — P2에서 confound를 제거한 방식. n_steps·prompt_len을 regress out하고 남은 구조 확인. 남으면 실제 신호, 사라지면 gentime은 형식만 인코딩.
- (B) hidden으로 전환 — gentime은 형식 아티팩트로 기록하고, hidden/pre-answer와 비교. 신호별 비교가 이 실험의 핵심.

In [ ]:
from sklearn.linear_model import LinearRegression

# n_steps + prompt_len을 gentime에서 regress out
conf = kn_labels[["n_steps","prompt_len"]].fillna(kn_labels[["n_steps","prompt_len"]].mean()).values
Xs = StandardScaler().fit_transform(gentime_kn)
resid = Xs - LinearRegression().fit(conf, Xs).predict(conf)  # confound 제거된 잔차

Xp_r = PCA(n_components=30, random_state=42).fit_transform(resid)
from sklearn.metrics import silhouette_score
print("=== n_steps/prompt_len 제거 후 gentime K- ===")
for k in [2,3,4]:
    km = KMeans(k, random_state=42, n_init=10).fit(Xp_r)
    sil = silhouette_score(Xp_r, km.labels_)
    # 이번엔 dataset/behavior 정렬도 바로
    ari_d = adjusted_rand_score(kn_labels["dataset_id"], km.labels_)
    ari_b = adjusted_rand_score(kn_labels["behavior"].fillna("NaN"), km.labels_)
    print(f"  k={k}: sil={sil:.3f} | dataset ARI={ari_d:+.3f} | behavior ARI={ari_b:+.3f}")

=== n_steps/prompt_len 제거 후 gentime K- ===
  k=2: sil=0.167 | dataset ARI=+0.028 | behavior ARI=+0.046
  k=3: sil=0.139 | dataset ARI=+0.035 | behavior ARI=+0.067
  k=4: sil=0.151 | dataset ARI=+0.037 | behavior ARI=+0.040


In [ ]:
hidden_kn = hidden[mask_kn]

# hidden은 고차원 → PCA 50으로
print("=== hidden K- 클러스터링 ===")
Xs = StandardScaler().fit_transform(hidden_kn)
pca = PCA(n_components=50, random_state=42).fit(Xs)
Xp = pca.transform(Xs)
print(f"PCA 50d 누적 설명분산: {pca.explained_variance_ratio_.cumsum()[-1]:.3f}")
print(f"상위10 누적: {pca.explained_variance_ratio_[:10].cumsum().round(3)}")

from sklearn.metrics import silhouette_score
for k in [2,3,4,5]:
    km = KMeans(k, random_state=42, n_init=10).fit(Xp)
    sil = silhouette_score(Xp, km.labels_)
    ari_d = adjusted_rand_score(kn_labels["dataset_id"], km.labels_)
    ari_b = adjusted_rand_score(kn_labels["behavior"].fillna("NaN"), km.labels_)
    ari_op = adjusted_rand_score(kn_labels["construction_operator"], km.labels_)
    print(f"  k={k}: sil={sil:.3f} | dataset ARI={ari_d:+.3f} | "
          f"behavior ARI={ari_b:+.3f} | operator ARI={ari_op:+.3f}")

# 길이 confound도 미리 체크
km2 = KMeans(2, random_state=42, n_init=10).fit(Xp)
print(f"\nk=2 길이 confound: prompt_len diff =",
      abs(kn_labels[km2.labels_==0]["prompt_len"].mean() - kn_labels[km2.labels_==1]["prompt_len"].mean()).round(1))
print(f"          n_steps diff =",
      abs(kn_labels[km2.labels_==0]["n_steps"].mean() - kn_labels[km2.labels_==1]["n_steps"].mean()).round(1))

=== hidden K- 클러스터링 ===
PCA 50d 누적 설명분산: 0.641
상위10 누적: [0.104 0.17  0.221 0.268 0.305 0.336 0.361 0.384 0.404 0.422]
  k=2: sil=0.144 | dataset ARI=+0.360 | behavior ARI=-0.010 | operator ARI=+0.267
  k=3: sil=0.145 | dataset ARI=+0.708 | behavior ARI=-0.002 | operator ARI=+0.525
  k=4: sil=0.127 | dataset ARI=+0.742 | behavior ARI=+0.006 | operator ARI=+0.604
  k=5: sil=0.150 | dataset ARI=+0.677 | behavior ARI=+0.009 | operator ARI=+0.563

k=2 길이 confound: prompt_len diff = 251.2
          n_steps diff = 8.1


In [ ]:
from sklearn.metrics import silhouette_score

print("=== within-dataset hidden 클러스터링 (K-, dataset 아티팩트 제거) ===")
for did in ["clasheval","confiqa_qa","dynamicqa","nqswap"]:
    m = mask_kn & (labels["dataset_id"]==did).values
    if m.sum() < 30: continue
    Hd = hidden[m]
    Xs = StandardScaler().fit_transform(Hd)
    Xp = PCA(n_components=min(30, m.sum()-1), random_state=42).fit_transform(Xs)
    ld = labels[m].reset_index(drop=True)
    for k in [2,3]:
        km = KMeans(k, random_state=42, n_init=10).fit(Xp)
        sil = silhouette_score(Xp, km.labels_)
        ari_b = adjusted_rand_score(ld["behavior"].fillna("NaN"), km.labels_)
        ari_op = adjusted_rand_score(ld["construction_operator"].fillna("NaN"), km.labels_)
        pl_diff = abs(ld[km.labels_==0]["prompt_len"].mean()-ld[km.labels_==1]["prompt_len"].mean()) if k==2 else 0
        print(f"  {did} (n={m.sum()}) k={k}: sil={sil:.3f} behavior_ARI={ari_b:+.3f} op_ARI={ari_op:+.3f}"
              + (f" len_diff={pl_diff:.0f}" if k==2 else ""))

=== within-dataset hidden 클러스터링 (K-, dataset 아티팩트 제거) ===
  clasheval (n=258) k=2: sil=0.166 behavior_ARI=+0.056 op_ARI=-0.021 len_diff=54
  clasheval (n=258) k=3: sil=0.222 behavior_ARI=+0.067 op_ARI=+0.159
  confiqa_qa (n=236) k=2: sil=0.112 behavior_ARI=+0.032 op_ARI=+0.000 len_diff=4
  confiqa_qa (n=236) k=3: sil=0.105 behavior_ARI=+0.159 op_ARI=+0.000
  dynamicqa (n=244) k=2: sil=0.181 behavior_ARI=+0.050 op_ARI=+0.001 len_diff=26
  dynamicqa (n=244) k=3: sil=0.206 behavior_ARI=+0.070 op_ARI=+0.009
  nqswap (n=230) k=2: sil=0.191 behavior_ARI=-0.022 op_ARI=+0.000 len_diff=234
  nqswap (n=230) k=3: sil=0.145 behavior_ARI=+0.010 op_ARI=+0.000


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

# hidden을 layer별로 reshape: [1916, 32, 4096]
hidden_3d = hidden.reshape(hidden.shape[0], 32, 4096)

# 타겟: K- 내부 accepted_context vs retained_parametric (이진)
kn = labels["K"]=="K-"
bin_mask = kn & labels["behavior"].isin(["accepted_context","retained_parametric"])
bin_mask = bin_mask.values
y = (labels[bin_mask]["behavior"]=="accepted_context").astype(int).values
groups = labels[bin_mask]["question"].values   # GroupKFold by question
ds_bin = labels[bin_mask]["dataset_id"].values
H = hidden_3d[bin_mask]

print(f"이진 타겟: accepted {y.sum()} vs retained {(1-y).sum()} (총 {len(y)})")
print("dataset 분포:", pd.Series(ds_bin).value_counts().to_dict())

def probe_layer(Hl, y, groups):
    """단일 layer, GroupKFold 5, AUROC 평균"""
    gkf = GroupKFold(n_splits=5)
    aucs = []
    for tr, te in gkf.split(Hl, y, groups):
        Xtr = StandardScaler().fit_transform(Hl[tr])
        Xte = StandardScaler().fit(Hl[tr]).transform(Hl[te])
        clf = LogisticRegression(max_iter=1000, C=0.1).fit(Xtr, y[tr])
        if len(np.unique(y[te]))>1:
            aucs.append(roc_auc_score(y[te], clf.predict_proba(Xte)[:,1]))
    return np.mean(aucs) if aucs else np.nan

# --- (1) pooled probe: layer별 (dataset shortcut 위험 있음, 비교용) ---
print("\n=== pooled probe (dataset shortcut 위험) ===")
pooled = {}
for L in [8, 12, 16, 18, 20, 24]:
    auc = probe_layer(H[:,L,:], y, groups)
    pooled[L] = auc
    print(f"  layer {L}: AUROC={auc:.3f}")

# --- (2) within-dataset probe: dataset 상수 → shortcut 차단 ---
print("\n=== within-dataset probe (layer 16) ===")
for did in ["clasheval","confiqa_qa","dynamicqa","nqswap"]:
    m = ds_bin==did
    if m.sum()<40 or len(np.unique(y[m]))<2:
        print(f"  {did}: 샘플 부족/단일클래스 (n={m.sum()}, accepted={y[m].sum()})")
        continue
    auc = probe_layer(H[m,16,:], y[m], groups[m])
    print(f"  {did} (n={m.sum()}, accepted율={y[m].mean():.2f}): AUROC={auc:.3f}")

이진 타겟: accepted 524 vs retained 83 (총 607)
dataset 분포: {'clasheval': 179, 'confiqa_qa': 147, 'dynamicqa': 142, 'nqswap': 139}

=== pooled probe (dataset shortcut 위험) ===
  layer 8: AUROC=0.730
  layer 12: AUROC=0.798
  layer 16: AUROC=0.819
  layer 18: AUROC=0.847
  layer 20: AUROC=0.853
  layer 24: AUROC=0.842

=== within-dataset probe (layer 16) ===
  clasheval (n=179, accepted율=0.91): AUROC=0.745
  confiqa_qa (n=147, accepted율=0.88): AUROC=0.845
  dynamicqa (n=142, accepted율=0.87): AUROC=0.810
  nqswap (n=139, accepted율=0.79): AUROC=0.758


In [ ]:
print("=== LODO: 3개 dataset 학습 → 1개 전이 (layer 20) ===")
L = 20
for held in ["clasheval","confiqa_qa","dynamicqa","nqswap"]:
    tr_m = ds_bin != held
    te_m = ds_bin == held
    if len(np.unique(y[te_m]))<2:
        print(f"  {held}: 홀드아웃 단일클래스, 스킵"); continue
    Xtr = StandardScaler().fit(H[tr_m,L,:])
    clf = LogisticRegression(max_iter=1000, C=0.1).fit(
        Xtr.transform(H[tr_m,L,:]), y[tr_m])
    auc = roc_auc_score(y[te_m], clf.predict_proba(Xtr.transform(H[te_m,L,:]))[:,1])
    print(f"  holdout={held} (n={te_m.sum()}): 전이 AUROC={auc:.3f}")

# layer 프로파일도 (P2 layer 16/18과 비교)
print("\n=== pooled layer 프로파일 (전체) ===")
for L in range(0, 32, 2):
    auc = probe_layer(H[:,L,:], y, groups)
    bar = "█"*int(auc*30)
    print(f"  L{L:2d}: {auc:.3f} {bar}")

=== LODO: 3개 dataset 학습 → 1개 전이 (layer 20) ===
  holdout=clasheval (n=179): 전이 AUROC=0.821
  holdout=confiqa_qa (n=147): 전이 AUROC=0.866
  holdout=dynamicqa (n=142): 전이 AUROC=0.894
  holdout=nqswap (n=139): 전이 AUROC=0.741

=== pooled layer 프로파일 (전체) ===
  L 0: 0.627 ██████████████████
  L 2: 0.704 █████████████████████
  L 4: 0.701 █████████████████████
  L 6: 0.706 █████████████████████
  L 8: 0.730 █████████████████████
  L10: 0.792 ███████████████████████
  L12: 0.798 ███████████████████████
  L14: 0.789 ███████████████████████
  L16: 0.819 ████████████████████████
  L18: 0.847 █████████████████████████
  L20: 0.853 █████████████████████████
  L22: 0.845 █████████████████████████
  L24: 0.842 █████████████████████████
  L26: 0.844 █████████████████████████
  L28: 0.855 █████████████████████████
  L30: 0.830 ████████████████████████


In [ ]:
from sklearn.utils import resample

# --- 1번: 홀드아웃별 소수 클래스 n ---
print("=== LODO 홀드아웃별 클래스 구성 ===")
for held in ["clasheval","confiqa_qa","dynamicqa","nqswap"]:
    m = ds_bin==held
    print(f"  {held}: retained={int((1-y[m]).sum())}, accepted={int(y[m].sum())}, n={m.sum()}")

# --- bootstrap CI 함수 ---
def lodo_auc_ci(held, L=20, n_boot=1000):
    tr_m = ds_bin != held; te_m = ds_bin == held
    sc = StandardScaler().fit(H[tr_m,L,:])
    clf = LogisticRegression(max_iter=1000, C=0.1).fit(sc.transform(H[tr_m,L,:]), y[tr_m])
    proba = clf.predict_proba(sc.transform(H[te_m,L,:]))[:,1]
    yte = y[te_m]
    point = roc_auc_score(yte, proba)
    boots = []
    idx = np.arange(len(yte))
    for _ in range(n_boot):
        bi = resample(idx, random_state=None)
        if len(np.unique(yte[bi]))>1:
            boots.append(roc_auc_score(yte[bi], proba[bi]))
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return point, lo, hi

print("\n=== LODO AUROC + 95% bootstrap CI (L20) ===")
for held in ["clasheval","confiqa_qa","dynamicqa","nqswap"]:
    p, lo, hi = lodo_auc_ci(held)
    print(f"  {held}: {p:.3f} [{lo:.3f}, {hi:.3f}]  (폭 {hi-lo:.3f})")

# --- 4번: hedged 제외 재실행 ---
print("\n=== hedged 제외 후 (bin_mask에서 hedged=True 빼고) ===")
hedged_arr = labels[bin_mask]["hedged"].fillna(False).values
nh = ~hedged_arr
print(f"hedged 제외: {nh.sum()}/{len(nh)} 남음 (제외 {hedged_arr.sum()})")
for L in [16, 20, 24]:
    auc = probe_layer(H[nh][:,L,:], y[nh], groups[nh])
    print(f"  layer {L} (hedged 제외): AUROC={auc:.3f}")

=== LODO 홀드아웃별 클래스 구성 ===
  clasheval: retained=17, accepted=162, n=179
  confiqa_qa: retained=18, accepted=129, n=147
  dynamicqa: retained=19, accepted=123, n=142
  nqswap: retained=29, accepted=110, n=139

=== LODO AUROC + 95% bootstrap CI (L20) ===
  clasheval: 0.821 [0.732, 0.896]  (폭 0.164)
  confiqa_qa: 0.866 [0.779, 0.934]  (폭 0.156)
  dynamicqa: 0.894 [0.811, 0.952]  (폭 0.141)
  nqswap: 0.741 [0.625, 0.830]  (폭 0.205)

=== hedged 제외 후 (bin_mask에서 hedged=True 빼고) ===
hedged 제외: 539/607 남음 (제외 68)
  layer 16 (hedged 제외): AUROC=0.847
  layer 20 (hedged 제외): AUROC=0.871
  layer 24 (hedged 제외): AUROC=0.844


In [ ]:
print("=== LODO 강건성: L16 / L24 (점추정만, 빠르게) ===")
for L in [16, 24]:
    print(f"\n--- layer {L} ---")
    for held in ["clasheval","confiqa_qa","dynamicqa","nqswap"]:
        tr_m = ds_bin!=held; te_m = ds_bin==held
        sc = StandardScaler().fit(H[tr_m,L,:])
        clf = LogisticRegression(max_iter=1000, C=0.1).fit(sc.transform(H[tr_m,L,:]), y[tr_m])
        auc = roc_auc_score(y[te_m], clf.predict_proba(sc.transform(H[te_m,L,:]))[:,1])
        print(f"  {held}: {auc:.3f}")

=== LODO 강건성: L16 / L24 (점추정만, 빠르게) ===

--- layer 16 ---
  clasheval: 0.730
  confiqa_qa: 0.884
  dynamicqa: 0.845
  nqswap: 0.722

--- layer 24 ---
  clasheval: 0.778
  confiqa_qa: 0.774
  dynamicqa: 0.861
  nqswap: 0.642


In [ ]:
from sklearn.metrics import silhouette_score

# DynamicQA K- 만 (한 출처 안에 static/temporal/disputable 공존)
m = mask_kn & (labels["dataset_id"]=="dynamicqa").values
ld = labels[m].reset_index(drop=True)
print("DynamicQA K- fact_nature 분포:", ld["fact_nature"].value_counts().to_dict())

# hidden으로 클러스터링 (dataset 상수 → 순수 fact_nature 테스트)
Hd = hidden[m]
Xs = StandardScaler().fit_transform(Hd)
Xp = PCA(n_components=min(30, m.sum()-1), random_state=42).fit_transform(Xs)

print("\n=== DynamicQA within-source 클러스터 vs fact_nature ===")
for k in [2, 3]:
    km = KMeans(k, random_state=42, n_init=10).fit(Xp)
    sil = silhouette_score(Xp, km.labels_)
    ari_fn = adjusted_rand_score(ld["fact_nature"], km.labels_)
    ari_beh = adjusted_rand_score(ld["behavior"].fillna("NaN"), km.labels_)
    print(f"  k={k}: sil={sil:.3f} | fact_nature ARI={ari_fn:+.3f} | behavior ARI={ari_beh:+.3f}")

# gentime으로도 (attention이 type 잡나)
print("\n=== DynamicQA gentime vs fact_nature ===")
Gd = gentime_arr[m]
Xs_g = StandardScaler().fit_transform(Gd)
Xp_g = PCA(n_components=min(30, m.sum()-1), random_state=42).fit_transform(Xs_g)
for k in [2, 3]:
    km = KMeans(k, random_state=42, n_init=10).fit(Xp_g)
    ari_fn = adjusted_rand_score(ld["fact_nature"], km.labels_)
    print(f"  k={k}: fact_nature ARI={ari_fn:+.3f}")

DynamicQA K- fact_nature 분포: {'static': 120, 'temporal_mutable': 94, 'disputable': 30}

=== DynamicQA within-source 클러스터 vs fact_nature ===
  k=2: sil=0.181 | fact_nature ARI=+0.067 | behavior ARI=+0.050
  k=3: sil=0.206 | fact_nature ARI=+0.156 | behavior ARI=+0.070

=== DynamicQA gentime vs fact_nature ===
  k=2: fact_nature ARI=+0.005
  k=3: fact_nature ARI=+0.089


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

# fact_nature를 이진으로: static vs temporal_mutable (K-, disputable 제외 — 소수)
type_mask = mask_kn & labels["fact_nature"].isin(["static","temporal_mutable"]).values
yt = (labels[type_mask]["fact_nature"]=="temporal_mutable").astype(int).values
gt = labels[type_mask]["question"].values
dt = labels[type_mask]["dataset_id"].values
Ht = hidden.reshape(hidden.shape[0],32,4096)[type_mask]

print(f"type 이진: temporal {yt.sum()} vs static {(1-yt).sum()} (총 {len(yt)})")
print("dataset 분포:", pd.Series(dt).value_counts().to_dict())

def probe_layer(Hl, y, groups):
    gkf = GroupKFold(n_splits=5); aucs=[]
    for tr,te in gkf.split(Hl,y,groups):
        sc=StandardScaler().fit(Hl[tr])
        clf=LogisticRegression(max_iter=1000,C=0.1).fit(sc.transform(Hl[tr]),y[tr])
        if len(np.unique(y[te]))>1:
            aucs.append(roc_auc_score(y[te],clf.predict_proba(sc.transform(Hl[te]))[:,1]))
    return np.mean(aucs) if aucs else np.nan

print("\n=== type probe (fact_nature) layer 프로파일 ===")
for L in [8,12,16,18,20,24]:
    print(f"  layer {L}: AUROC={probe_layer(Ht[:,L,:],yt,gt):.3f}")

# ⚠️ type이 dataset에 강하게 nested (temporal≈dynamicqa, static은 여러곳)
# → within-dataset에서도 봐야 shortcut 배제. dynamicqa는 static+temporal 둘 다 있음
print("\n=== type probe within dynamicqa (dataset shortcut 배제) ===")
m_dyn = dt=="dynamicqa"
if len(np.unique(yt[m_dyn]))>1:
    for L in [16,20]:
        print(f"  layer {L}: AUROC={probe_layer(Ht[m_dyn,L,:],yt[m_dyn],gt[m_dyn]):.3f}")

print(f"\n=== 대비 ===")
print(f"behavior probe (accepted vs retained): AUROC 0.85 [확보됨]")
print(f"type probe (temporal vs static): 위 결과와 비교")

type 이진: temporal 94 vs static 844 (총 938)
dataset 분포: {'clasheval': 258, 'confiqa_qa': 236, 'nqswap': 230, 'dynamicqa': 214}

=== type probe (fact_nature) layer 프로파일 ===
  layer 8: AUROC=0.953
  layer 12: AUROC=0.964
  layer 16: AUROC=0.958
  layer 18: AUROC=0.960
  layer 20: AUROC=0.956
  layer 24: AUROC=0.954

=== type probe within dynamicqa (dataset shortcut 배제) ===
  layer 16: AUROC=0.756
  layer 20: AUROC=0.744

=== 대비 ===
behavior probe (accepted vs retained): AUROC 0.85 [확보됨]
type probe (temporal vs static): 위 결과와 비교


In [ ]:
from sklearn.utils import resample

# within-dynamicqa type probe CI + 클래스 수
m_dyn = dt=="dynamicqa"
print("dynamicqa 내 type:", "temporal", int(yt[m_dyn].sum()), "static", int((1-yt[m_dyn]).sum()))

# GroupKFold 대신 전체 CV point + bootstrap on predictions
def probe_ci(Hl, y, groups, L, n_boot=1000):
    gkf = GroupKFold(n_splits=5)
    all_true, all_prob = [], []
    for tr,te in gkf.split(Hl,y,groups):
        sc=StandardScaler().fit(Hl[tr])
        clf=LogisticRegression(max_iter=1000,C=0.1).fit(sc.transform(Hl[tr]),y[tr])
        all_true.extend(y[te]); all_prob.extend(clf.predict_proba(sc.transform(Hl[te]))[:,1])
    all_true, all_prob = np.array(all_true), np.array(all_prob)
    point = roc_auc_score(all_true, all_prob)
    boots=[]
    for _ in range(n_boot):
        bi=resample(np.arange(len(all_true)))
        if len(np.unique(all_true[bi]))>1:
            boots.append(roc_auc_score(all_true[bi], all_prob[bi]))
    return point, np.percentile(boots,[2.5,97.5])

for L in [16,20]:
    p,(lo,hi)=probe_ci(Ht[m_dyn,L,:],yt[m_dyn],gt[m_dyn],L)
    print(f"  within-dynamicqa L{L}: {p:.3f} [{lo:.3f},{hi:.3f}]")

# 비교용: behavior probe도 같은 방식 CI (accepted vs retained)
print("\n비교 — behavior probe CI:")
p,(lo,hi)=probe_ci(H[:,20,:],y,groups,20)
print(f"  behavior L20: {p:.3f} [{lo:.3f},{hi:.3f}]")

dynamicqa 내 type: temporal 94 static 120
  within-dynamicqa L16: 0.750 [0.682,0.814]
  within-dynamicqa L20: 0.732 [0.666,0.792]

비교 — behavior probe CI:
  behavior L20: 0.843 [0.797,0.885]


In [ ]:
# gentime [1916, 3072] = [layer, head, 3구간(ctx/q/prevgen)]
# prev-gen 구간 빼고 ctx+q만, (ctx+q)로 재정규화 → n_steps 상관 제거
g3 = gentime_arr.reshape(1916, 32, 32, 3)
ctx_q = g3[:, :, :, :2]  # ctx, q만 (prev-gen 제거)
# ctx+q 합으로 재정규화 (각 head에서 ctx/(ctx+q), q/(ctx+q))
denom = ctx_q.sum(axis=3, keepdims=True) + 1e-9
ctx_q_norm = ctx_q / denom
gentime_renorm = ctx_q_norm.reshape(1916, 32*32*2)  # [1916, 2048]

print("재정규화 gentime:", gentime_renorm.shape)

# K- 재클러스터 + n_steps 상관 확인
gr_kn = gentime_renorm[mask_kn]
Xs = StandardScaler().fit_transform(gr_kn)
Xp = PCA(n_components=30, random_state=42).fit_transform(Xs)
from sklearn.metrics import silhouette_score
print("\n=== 재정규화 gentime K- ===")
for k in [2,3]:
    km = KMeans(k, random_state=42, n_init=10).fit(Xp)
    sil = silhouette_score(Xp, km.labels_)
    ari_d = adjusted_rand_score(kn_labels["dataset_id"], km.labels_)
    ari_b = adjusted_rand_score(kn_labels["behavior"].fillna("NaN"), km.labels_)
    ns_diff = abs(kn_labels[km.labels_==0]["n_steps"].mean()-kn_labels[km.labels_==1]["n_steps"].mean()) if k==2 else 0
    print(f"  k={k}: sil={sil:.3f} dataset_ARI={ari_d:+.3f} behavior_ARI={ari_b:+.3f}"
          + (f" n_steps_diff={ns_diff:.1f}" if k==2 else ""))

재정규화 gentime: (1916, 2048)

=== 재정규화 gentime K- ===
  k=2: sil=0.286 dataset_ARI=+0.024 behavior_ARI=+0.011 n_steps_diff=13.0
  k=3: sil=0.235 dataset_ARI=+0.052 behavior_ARI=+0.016


In [ ]:
print("=== behavior별 n_steps (길이가 행동의 그림자인지) ===")
print(kn_labels.groupby("behavior")["n_steps"].agg(["mean","median","count"]))

# 통계적으로 갈리는지 (분산분석 간이)
from scipy.stats import kruskal
groups_ns = [kn_labels[kn_labels["behavior"]==b]["n_steps"].values
             for b in kn_labels["behavior"].unique()]
h, p = kruskal(*groups_ns)
print(f"\nKruskal-Wallis: H={h:.1f}, p={p:.2e}")
print("→ behavior 간 n_steps 차이:", "유의 (길이=행동 프록시)" if p<0.01 else "무의미 (순수 길이 아티팩트)")

=== behavior별 n_steps (길이가 행동의 그림자인지) ===
                          mean  median  count
behavior                                     
abstain              31.703297    24.0     91
accepted_context     30.133588    26.5    524
both                 41.333333    37.0     87
other                32.043716    28.0    183
retained_parametric  31.373494    25.0     83

Kruskal-Wallis: H=30.8, p=3.32e-06
→ behavior 간 n_steps 차이: 유의 (길이=행동 프록시)


In [ ]:
# hidden probe와 동일 타겟/방식 (accepted vs retained, GroupKFold)
# gentime은 layer 구조가 [32L×32H×3]이라 통째 2048d(재정규화) or 3072d(원본) 둘 다
G_bin_raw = gentime_arr[bin_mask]        # 3072d 원본
G_bin_norm = gentime_renorm[bin_mask]    # 2048d 재정규화(ctx/q만)

def probe_flat(X, y, groups):
    gkf = GroupKFold(n_splits=5); aucs=[]
    for tr,te in gkf.split(X,y,groups):
        sc=StandardScaler().fit(X[tr])
        clf=LogisticRegression(max_iter=1000,C=0.1).fit(sc.transform(X[tr]),y[tr])
        if len(np.unique(y[te]))>1:
            aucs.append(roc_auc_score(y[te],clf.predict_proba(sc.transform(X[te]))[:,1]))
    return np.mean(aucs) if aucs else np.nan

print("=== gentime supervised probe (accepted vs retained) ===")
print(f"  gentime 원본 3072d:      AUROC={probe_flat(G_bin_raw, y, groups):.3f}")
print(f"  gentime 재정규화 2048d:  AUROC={probe_flat(G_bin_norm, y, groups):.3f}")
print(f"\n  [비교] hidden L20:       AUROC=0.843")

# n_steps 단독으로도 예측되나 (baseline — 길이만으로 얼마나?)
ns = kn_labels.loc[bin_mask[mask_kn], "n_steps"].values if bin_mask[mask_kn].sum()==len(y) else None
# 간단히: bin_mask 샘플의 n_steps
ns_bin = labels[bin_mask]["n_steps"].values.reshape(-1,1)
print(f"\n  [baseline] n_steps 단독:  AUROC={probe_flat(ns_bin, y, groups):.3f}")

=== gentime supervised probe (accepted vs retained) ===
  gentime 원본 3072d:      AUROC=0.865
  gentime 재정규화 2048d:  AUROC=0.844

  [비교] hidden L20:       AUROC=0.843

  [baseline] n_steps 단독:  AUROC=0.474


In [ ]:
print(kn_labels.groupby("behavior")["n_steps"].agg(["mean","median","count"]))

                          mean  median  count
behavior                                     
abstain              31.703297    24.0     91
accepted_context     30.133588    26.5    524
both                 41.333333    37.0     87
other                32.043716    28.0    183
retained_parametric  31.373494    25.0     83


In [ ]:
# pre-answer는 128d (32L × 4구간metric). bin_mask로 같은 타겟
P_bin = preanswer[bin_mask]  # [n, 128]

print("=== pre-answer supervised probe (accepted vs retained) ===")
print(f"  pre-answer 128d: AUROC={probe_flat(P_bin, y, groups):.3f}")
print(f"\n  [비교]")
print(f"  gentime 재정규화: 0.844")
print(f"  hidden L20:       0.843")
print(f"  n_steps baseline: 0.474")

# pre-answer도 unsupervised에선 뭘 잡았나 (K- 클러스터)
P_kn = preanswer[mask_kn]
Xs = StandardScaler().fit_transform(P_kn)
Xp = PCA(n_components=min(30,128), random_state=42).fit_transform(Xs)
from sklearn.metrics import silhouette_score
print(f"\n=== pre-answer K- 클러스터 (unsupervised) ===")
for k in [2,3]:
    km = KMeans(k, random_state=42, n_init=10).fit(Xp)
    sil = silhouette_score(Xp, km.labels_)
    ari_d = adjusted_rand_score(kn_labels["dataset_id"], km.labels_)
    ari_b = adjusted_rand_score(kn_labels["behavior"].fillna("NaN"), km.labels_)
    ns = abs(kn_labels[km.labels_==0]["n_steps"].mean()-kn_labels[km.labels_==1]["n_steps"].mean()) if k==2 else 0
    pl = abs(kn_labels[km.labels_==0]["prompt_len"].mean()-kn_labels[km.labels_==1]["prompt_len"].mean()) if k==2 else 0
    print(f"  k={k}: sil={sil:.3f} dataset_ARI={ari_d:+.3f} behavior_ARI={ari_b:+.3f}"
          + (f" prompt_len_diff={pl:.0f}" if k==2 else ""))

=== pre-answer supervised probe (accepted vs retained) ===
  pre-answer 128d: AUROC=0.759

  [비교]
  gentime 재정규화: 0.844
  hidden L20:       0.843
  n_steps baseline: 0.474

=== pre-answer K- 클러스터 (unsupervised) ===
  k=2: sil=0.314 dataset_ARI=+0.273 behavior_ARI=-0.012 prompt_len_diff=274
  k=3: sil=0.318 dataset_ARI=+0.293 behavior_ARI=+0.002


In [ ]:
g_ha = gentime_renorm.reshape(-1,32,32,2).mean(axis=2).reshape(-1,64)  # head 평균 64d
print("gentime head-avg probe:", probe_flat(g_ha[bin_mask], y, groups))

gentime head-avg probe: 0.6717764730469543


In [ ]:
@torch.no_grad()
def extract_preanswer_perhead(question, context):
    """pre-answer attention, head 유지. ctx/q mass를 head별로 → [32L × 32H × 2]"""
    b = get_segment_bounds_neutral(question, context)
    full = build_neutral_prompt(question, context)
    enc = tok(full, return_tensors="pt", add_special_tokens=False).to("cuda")
    out = model(**enc, output_attentions=True, use_cache=False)
    ctx_s, ctx_e = b["ctx_range"]; q_s, q_e = b["q_range"]
    last = enc["input_ids"].shape[1] - 1

    feats = np.zeros((32, 32, 2), dtype=np.float32)
    for L in range(len(out.attentions)):
        a = out.attentions[L][0][:, last, :]        # [head, kv] — .mean(0) 제거
        ctx_mass = a[:, ctx_s:ctx_e].sum(-1)        # [head]
        q_mass = a[:, q_s:q_e].sum(-1)
        denom = ctx_mass + q_mass + 1e-9
        feats[L, :, 0] = (ctx_mass/denom).float().cpu().numpy()  # ctx 비율 (재정규화)
        feats[L, :, 1] = (q_mass/denom).float().cpu().numpy()
    return feats.reshape(-1)  # [2048]

t0 = time.time()
pre_ph = []
for _, ex in tqdm(strat_new.iterrows(), total=len(strat_new)):
    pre_ph.append(extract_preanswer_perhead(ex["question"], ex["context_trunc"]))
print(f"완료: {(time.time()-t0)/60:.1f}분")

pre_perhead = np.stack(pre_ph)  # [1916, 2048]
np.save("preanswer_perhead.npy", pre_perhead)
print("shape:", pre_perhead.shape)

# probe (공정 비교: per-head vs per-head)
P_ph_bin = pre_perhead[bin_mask]
print(f"\n=== 공정 비교 (per-head 통일) ===")
print(f"  pre-answer per-head 2048d: AUROC={probe_flat(P_ph_bin, y, groups):.3f}")
print(f"  gentime per-head 3072d:    0.865 [기확보]")
print(f"  hidden L20:                0.843 [기확보]")

  0%|          | 0/1916 [00:00<?, ?it/s]

완료: 4.5분
shape: (1916, 2048)

=== 공정 비교 (per-head 통일) ===
  pre-answer per-head 2048d: AUROC=0.810
  gentime per-head 3072d:    0.865 [기확보]
  hidden L20:                0.843 [기확보]


In [ ]:
from sklearn.utils import resample

def probe_ci_flat(X, y, groups, n_boot=1000):
    gkf = GroupKFold(n_splits=5)
    all_true, all_prob = [], []
    for tr,te in gkf.split(X,y,groups):
        sc=StandardScaler().fit(X[tr])
        clf=LogisticRegression(max_iter=1000,C=0.1).fit(sc.transform(X[tr]),y[tr])
        all_true.extend(y[te]); all_prob.extend(clf.predict_proba(sc.transform(X[te]))[:,1])
    at,ap=np.array(all_true),np.array(all_prob)
    point=roc_auc_score(at,ap)
    boots=[roc_auc_score(at[bi],ap[bi]) for bi in
           (resample(np.arange(len(at))) for _ in range(n_boot))
           if len(np.unique(at[resample(np.arange(len(at)),random_state=None)]))>1]
    # 안전한 bootstrap
    boots=[]
    for _ in range(n_boot):
        bi=resample(np.arange(len(at)))
        if len(np.unique(at[bi]))>1: boots.append(roc_auc_score(at[bi],ap[bi]))
    return point, np.percentile(boots,[2.5,97.5])

print("=== predictive 신호 공정 비교 + CI ===")
for name, X in [("hidden L20", H[:,20,:]),
                ("pre-answer per-head", pre_perhead[bin_mask])]:
    p,(lo,hi)=probe_ci_flat(X, y, groups)
    print(f"  {name:22s}: {p:.3f} [{lo:.3f}, {hi:.3f}]")

=== predictive 신호 공정 비교 + CI ===
  hidden L20            : 0.843 [0.795, 0.883]
  pre-answer per-head   : 0.793 [0.741, 0.846]


In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.utils import resample

def get_oof_preds(X, y, groups, splits):
    """주어진 분할(splits)로 out-of-fold 예측 반환"""
    oof = np.zeros(len(y))
    for tr, te in splits:
        sc = StandardScaler().fit(X[tr])
        clf = LogisticRegression(max_iter=1000, C=0.1).fit(sc.transform(X[tr]), y[tr])
        oof[te] = clf.predict_proba(sc.transform(X[te]))[:, 1]
    return oof

# 동일 분할 고정 (두 신호 공유)
splits = list(GroupKFold(n_splits=5).split(H[:,20,:], y, groups))

# 같은 분할로 두 신호 OOF 예측
ap_hidden = get_oof_preds(H[:,20,:], y, groups, splits)
ap_preans = get_oof_preds(pre_perhead[bin_mask], y, groups, splits)
at = y  # 정답 (bin_mask 적용된 y)

# paired bootstrap delta
delta = []
rng = np.arange(len(at))
for _ in range(1000):
    bi = resample(rng)
    if len(np.unique(at[bi])) > 1:
        delta.append(roc_auc_score(at[bi], ap_hidden[bi]) - roc_auc_score(at[bi], ap_preans[bi]))

lo, hi = np.percentile(delta, [2.5, 97.5])
print(f"hidden AUROC: {roc_auc_score(at, ap_hidden):.3f}")
print(f"pre-answer AUROC: {roc_auc_score(at, ap_preans):.3f}")
print(f"ΔAUROC (hidden - preanswer): {np.mean(delta):+.3f} [{lo:+.3f}, {hi:+.3f}]")
print("→", "차이 유의 (0 미포함)" if lo > 0 or hi < 0 else "차이 무의미 (0 포함) = 두 신호 동등")

hidden AUROC: 0.843
pre-answer AUROC: 0.793
ΔAUROC (hidden - preanswer): +0.049 [+0.006, +0.098]
→ 차이 유의 (0 미포함)


In [ ]:
import os
files_to_get = [
    # 데이터/라벨
    "strat_v2_final.parquet",
    "strat_v2_behavior.parquet",
    "label_provenance.json",
    "conflict_pool_kstate.parquet",
    # 신호
    "hidden_features.npy",
    "gentime_attention.npy",
    "preanswer_attention.npy",
    "preanswer_perhead.npy",
    "signal_sample_ids.npy",
    "gentime_sample_ids.npy",
    "gentime_meta.parquet",
]

print("=== 존재 확인 ===")
for f in files_to_get:
    if os.path.exists(f):
        sz = os.path.getsize(f)/1e6
        print(f"  ✓ {f} ({sz:.1f} MB)")
    else:
        print(f"  ✗ {f} — 없음")

=== 존재 확인 ===
  ✓ strat_v2_final.parquet (3.4 MB)
  ✓ strat_v2_behavior.parquet (3.7 MB)
  ✓ label_provenance.json (0.0 MB)
  ✓ conflict_pool_kstate.parquet (19.7 MB)
  ✓ hidden_features.npy (1004.5 MB)
  ✓ gentime_attention.npy (23.5 MB)
  ✓ preanswer_attention.npy (1.0 MB)
  ✓ preanswer_perhead.npy (15.7 MB)
  ✓ signal_sample_ids.npy (0.2 MB)
  ✓ gentime_sample_ids.npy (0.0 MB)
  ✓ gentime_meta.parquet (0.2 MB)


In [ ]:
from google.colab import files
for f in files_to_get:
    if os.path.exists(f):
        files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>